# SemCons-IDS — MiniLM vs ModernBERT Comparative Resume-Safe Edition v9

Cette version reprend automatiquement une campagne interrompue.

- Les runs déjà terminés sont rechargés depuis Google Drive.
- Les entraînements terminés ne sont jamais répétés.
- Seuls les runs incomplets ou absents sont exécutés.
- Les tableaux, conformal analyses, explications et figures sont ensuite reconstruits avec l'ensemble des runs.

## 0 — Installation compatible Google Colab

In [ ]:
import os
import sys
import signal
import subprocess
import importlib.metadata as metadata

PINNED = {
    "transformers": "4.57.6",
    "peft": "0.18.0",
    "accelerate": "1.12.0",
    "xgboost": "2.1.4",
}

def version_of(name: str):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None

def stack_ok() -> bool:
    code = (
        "from transformers import AutoTokenizer, AutoModelForSequenceClassification;"
        "from peft import LoraConfig, TaskType, get_peft_model;"
        "from xgboost import XGBClassifier;"
        "print('STACK_OK')"
    )
    result = subprocess.run(
        [sys.executable, "-c", code],
        text=True,
        capture_output=True,
    )
    return result.returncode == 0 and "STACK_OK" in result.stdout

installed = {name: version_of(name) for name in PINNED}
torchao_version = version_of("torchao")

needs_repair = (
    torchao_version is not None
    or any(installed[name] != expected for name, expected in PINNED.items())
    or not stack_ok()
)

print("Versions avant vérification :")
for name, expected in PINNED.items():
    print(f"  {name:14s}: {installed[name]} (attendu : {expected})")
print(f"  {'torchao':14s}: {torchao_version or 'absent'}")

if needs_repair:
    print("\nRéinstallation propre de la pile...")
    subprocess.run(
        [
            sys.executable, "-m", "pip", "uninstall", "-y",
            "transformers", "peft", "accelerate", "tokenizers", "torchao", "xgboost",
        ],
        check=False,
    )
    subprocess.check_call(
        [
            sys.executable, "-m", "pip", "install", "--no-cache-dir",
            "transformers==4.57.6",
            "peft==0.18.0",
            "accelerate==1.12.0",
            "xgboost==2.1.4",
            "pyarrow>=15.0.0",
            "imbalanced-learn>=0.12.0",
            "joblib>=1.3.0",
            "psutil>=5.9.0",
            "matplotlib>=3.8.0",
            "seaborn>=0.13.2",
        ]
    )
    if not stack_ok():
        raise RuntimeError("La pile réinstallée n'est pas importable.")
    print("✅ Installation réparée.")
    print("♻️ Redémarrage automatique du noyau...")
    os.kill(os.getpid(), signal.SIGKILL)

print("✅ Environnement stable et prêt.")

## 1 — Imports, configuration et dossiers de sortie

In [ ]:
from __future__ import annotations

import gc
import json
import math
import os
import random
import re
import time
import warnings
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Dict, List, Tuple

import importlib.metadata as metadata
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb

from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
    matthews_corrcoef,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForSeq2SeqLM
from peft import LoraConfig, TaskType, get_peft_model, PeftModel
from xgboost import XGBClassifier

try:
    from scipy.stats import wilcoxon
except Exception:
    wilcoxon = None

warnings.filterwarnings("ignore")
sns.set_context("talk")

EXPECTED = {
    "transformers": "4.57.6",
    "peft": "0.18.0",
    "accelerate": "1.12.0",
    "xgboost": "2.1.4",
}
for package, expected in EXPECTED.items():
    found = metadata.version(package)
    if found != expected:
        raise RuntimeError(
            f"{package}={found}; attendu={expected}. "
            "Réexécutez la cellule d'installation et laissez le noyau redémarrer."
        )

@dataclass
class Config:
    data_dir: str = "/content/drive/MyDrive/Dataset/CIC-IDS-2017/"
    output_dir: str = "/content/drive/MyDrive/these_llm/SemCons_IDS_MiniLM_ModernBERT_v8/"
    model_checkpoint: str = "sentence-transformers/all-MiniLM-L6-v2"
    current_backbone: str = "minilm"
    backbones: List[str] = field(default_factory=lambda: ["minilm", "modernbert"])
    lora_target_modules: List[str] = field(default_factory=lambda: ["query", "value"])

    seeds: List[int] = field(default_factory=lambda: [42, 123, 2026])
    protocols: List[str] = field(default_factory=lambda: ["iid_stratified", "temporal_within_file"])

    train_fraction: float = 0.70
    val_fraction: float = 0.10
    cal_fraction: float = 0.10
    test_fraction: float = 0.10

    min_samples_per_class: int = 200
    numeric_top_k: int = 60
    text_top_k: int = 16
    feature_selection_rows_per_class: int = 20_000

    llm_train_cap_per_class: int = 20_000
    llm_val_cap_per_class: int = 5_000
    max_length: int = 320

    lora_r: int = 32
    lora_alpha: int = 64
    lora_dropout: float = 0.05

    epochs: int = 8
    learning_rate: float = 1e-4
    weight_decay: float = 0.01
    batch_train: int = 32
    batch_eval: int = 128
    grad_accum: int = 1
    warmup_fraction: float = 0.06
    label_smoothing: float = 0.01
    early_stopping_patience: int = 2

    use_sqrt_class_weights: bool = True
    class_weight_min: float = 0.60
    class_weight_max: float = 2.50

    xgb_n_estimators: int = 1200
    xgb_max_depth: int = 12
    xgb_learning_rate: float = 0.08
    xgb_subsample: float = 0.90
    xgb_colsample_bytree: float = 0.90
    xgb_min_child_weight: float = 1.0
    xgb_reg_lambda: float = 1.0
    xgb_early_stopping_rounds: int = 50

    fusion_selection_metric: str = "weighted_f1"
    fusion_min_semantic_weight: float = 0.00
    fusion_grid_points: int = 41

    target_weighted_f1: float = 0.99
    selection_gain_margin: float = 5e-4

    conformal_alpha: float = 0.10
    raps_lambda: float = 0.02
    raps_k_reg: int = 2
    conformal_abstain_size: int = 2

    explanation_model_checkpoint: str = "google/flan-t5-small"
    explanation_samples: int = 60
    slm_explanation_samples: int = 30
    explanation_top_k: int = 5
    explanation_max_new_tokens: int = 96
    explanation_language: str = "English"

    figure_dpi: int = 220
    resume_completed_runs: bool = True
    force_rerun: bool = False

CFG = Config()


BACKBONE_SPECS = {
    "minilm": {
        "display_name": "MiniLM",
        "checkpoint": "sentence-transformers/all-MiniLM-L6-v2",
        "target_modules": ["query", "value"],
        "modules_to_save": ["classifier"],
        "batch_train": 32,
        "batch_eval": 128,
        "grad_accum": 1,
        "expected_parameters_m": 22.7,
    },
    "modernbert": {
        "display_name": "ModernBERT",
        "checkpoint": "answerdotai/ModernBERT-base",
        "target_modules": ["Wqkv"],
        "modules_to_save": ["classifier"],
        "batch_train": 16,
        "batch_eval": 64,
        "grad_accum": 2,
        "expected_parameters_m": 149.0,
    },
}

def apply_backbone_config(backbone: str) -> Dict[str, object]:
    if backbone not in BACKBONE_SPECS:
        raise KeyError(f"Backbone inconnu : {backbone}")
    spec = BACKBONE_SPECS[backbone]
    CFG.current_backbone = backbone
    CFG.model_checkpoint = str(spec["checkpoint"])
    CFG.lora_target_modules = list(spec["target_modules"])
    CFG.batch_train = int(spec["batch_train"])
    CFG.batch_eval = int(spec["batch_eval"])
    CFG.grad_accum = int(spec["grad_accum"])
    return spec

def backbone_display_name(backbone: str) -> str:
    return str(BACKBONE_SPECS[backbone]["display_name"])

apply_backbone_config(CFG.backbones[0])


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(CFG.seeds[0])

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print("Drive non monté :", exc)

OUTPUT_DIR = Path(CFG.output_dir)
FIG_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
MODEL_DIR = OUTPUT_DIR / "models"
for directory in [OUTPUT_DIR, FIG_DIR, TABLE_DIR, MODEL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(json.dumps(asdict(CFG), indent=2, ensure_ascii=False))
print("Device :", DEVICE)
if DEVICE.type == "cuda":
    print("GPU    :", torch.cuda.get_device_name(0))
    print("VRAM   :", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2), "GiB")

plt.rcParams["figure.dpi"] = CFG.figure_dpi
plt.rcParams["savefig.dpi"] = CFG.figure_dpi

def save_figure(filename_stem: str) -> None:
    png = FIG_DIR / f"{filename_stem}.png"
    pdf = FIG_DIR / f"{filename_stem}.pdf"
    plt.tight_layout()
    plt.savefig(png, bbox_inches="tight")
    plt.savefig(pdf, bbox_inches="tight")
    print("Figure sauvegardée :", png)

## 2 — Chargement, normalisation des labels et déduplication

In [ ]:
def normalize_column_name(name: str) -> str:
    value = str(name).strip().lower()
    value = re.sub(r"[\s/]+", "_", value)
    value = re.sub(r"[^a-z0-9_]", "", value)
    value = re.sub(r"_+", "_", value)
    return value.strip("_")

def normalize_label(value: object) -> str | None:
    raw = str(value).strip()
    key = re.sub(r"\s+", " ", raw.lower().replace("_", " ")).strip()
    mapping = {
        "benign": "BENIGN",
        "bot": "Bot",
        "ddos": "DDoS",
        "dos hulk": "DoS_Hulk",
        "dos goldeneye": "DoS_GoldenEye",
        "dos slowhttptest": "DoS_Slowhttptest",
        "dos slowloris": "DoS_Slowloris",
        "ftp-patator": "FTP-Patator",
        "ftp patator": "FTP-Patator",
        "ssh-patator": "SSH-Patator",
        "ssh patator": "SSH-Patator",
        "portscan": "PortScan",
        "port scan": "PortScan",
        "infiltration": "Infiltration",
        "heartbleed": "Heartbleed",
    }
    return mapping.get(key)

def find_data_files(data_dir: str) -> List[Path]:
    root = Path(data_dir)
    files = []
    for pattern in ("*.parquet", "*.csv", "*.feather"):
        files.extend(root.glob(pattern))
    return sorted(files)

def read_one_file(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix == ".parquet":
        frame = pd.read_parquet(path)
    elif suffix == ".feather":
        frame = pd.read_feather(path)
    elif suffix == ".csv":
        frame = pd.read_csv(path, low_memory=False)
    else:
        raise ValueError(path)
    frame.columns = [normalize_column_name(c) for c in frame.columns]
    frame["__source_file__"] = path.name
    frame["__row_in_file__"] = np.arange(len(frame))
    return frame

files = find_data_files(CFG.data_dir)
if not files:
    raise FileNotFoundError(f"Aucun CSV/Parquet/Feather dans {CFG.data_dir}")

frames = []
for file_path in files:
    frame = read_one_file(file_path)
    frames.append(frame)
    print(f"{file_path.name:<70s} {len(frame):>10,} lignes")

df = pd.concat(frames, ignore_index=True)

label_candidates = [c for c in df.columns if c == "label" or "label" in c or "class" in c]
if not label_candidates:
    raise ValueError("Colonne label introuvable.")
LABEL_COL = label_candidates[0]

df[LABEL_COL] = df[LABEL_COL].map(normalize_label)
unmapped = int(df[LABEL_COL].isna().sum())
df = df.dropna(subset=[LABEL_COL]).copy()

before_dedup = len(df)
dedup_columns = [c for c in df.columns if c not in {"__source_file__", "__row_in_file__"}]
df = df.drop_duplicates(subset=dedup_columns).reset_index(drop=True)

feature_candidates = [
    c for c in df.columns
    if c not in {LABEL_COL, "__source_file__", "__row_in_file__"}
]
for column in feature_candidates:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df[feature_candidates] = df[feature_candidates].replace([np.inf, -np.inf], np.nan)

counts = df[LABEL_COL].value_counts()
keep_classes = counts[counts >= CFG.min_samples_per_class].index.tolist()
removed_classes = sorted(set(counts.index) - set(keep_classes))
df = df[df[LABEL_COL].isin(keep_classes)].reset_index(drop=True)

print(f"\nLabels non reconnus supprimés : {unmapped:,}")
print(f"Doublons supprimés            : {before_dedup - len(df):,}")
print(f"Dataset retenu                : {len(df):,} lignes × {len(feature_candidates)} variables candidates")
print("Classes supprimées            :", removed_classes)
display(df[LABEL_COL].value_counts().to_frame("effectif"))

ALL_CLASS_COUNTS = df[LABEL_COL].value_counts().rename_axis("label").reset_index(name="count")
ALL_CLASS_COUNTS.to_csv(TABLE_DIR / "overall_class_distribution.csv", index=False)

## 3 — Graphe global : distribution des classes

In [ ]:
distribution_plot_df = ALL_CLASS_COUNTS.sort_values("count", ascending=False)

plt.figure(figsize=(12, 5))
plt.bar(distribution_plot_df["label"], distribution_plot_df["count"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Nombre de flux")
plt.title("Distribution globale des classes — CICIDS2017 nettoyé")
save_figure("global_class_distribution")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import numpy as np

# ============================================================
# Distribution globale des classes — CICIDS2017
# ============================================================

distribution_plot_df = (
    ALL_CLASS_COUNTS
    .sort_values("count", ascending=True)
    .reset_index(drop=True)
    .copy()
)

# Calcul des proportions
total_flows = distribution_plot_df["count"].sum()
distribution_plot_df["percentage"] = (
    100 * distribution_plot_df["count"] / total_flows
)

# Palette progressive
colors = plt.cm.Blues(
    np.linspace(0.40, 0.90, len(distribution_plot_df))
)

fig, ax = plt.subplots(figsize=(11, 6.5))

bars = ax.barh(
    distribution_plot_df["label"],
    distribution_plot_df["count"],
    color=colors,
    edgecolor="white",
    linewidth=0.8,
    height=0.72
)

# Ajouter l'effectif et le pourcentage à droite des barres
max_count = distribution_plot_df["count"].max()

for bar, count, percentage in zip(
    bars,
    distribution_plot_df["count"],
    distribution_plot_df["percentage"]
):
    ax.text(
        bar.get_width() + max_count * 0.012,
        bar.get_y() + bar.get_height() / 2,
        f"{count:,.0f}  ({percentage:.2f} %)",
        va="center",
        ha="left",
        fontsize=9,
        fontweight="medium"
    )

# Format compact de l'axe x : 1 k, 10 k, 1 M, etc.
def compact_number(value, _):
    if value >= 1_000_000:
        return f"{value / 1_000_000:.1f} M"
    if value >= 1_000:
        return f"{value / 1_000:.0f} k"
    return f"{value:.0f}"

ax.xaxis.set_major_formatter(FuncFormatter(compact_number))

# Mise en forme
ax.set_title(
    "Distribution globale des classes dans CICIDS2017 nettoyé",
    fontsize=15,
    fontweight="bold",
    pad=16
)

ax.set_xlabel(
    "Nombre de flux réseau",
    fontsize=11,
    labelpad=10
)

ax.set_ylabel("")

ax.grid(
    axis="x",
    linestyle="--",
    linewidth=0.7,
    alpha=0.35
)

ax.set_axisbelow(True)

# Retirer les bordures inutiles
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)

ax.tick_params(
    axis="y",
    labelsize=10,
    length=0
)

ax.tick_params(
    axis="x",
    labelsize=9
)

# Espace pour les annotations
ax.set_xlim(0, max_count * 1.23)

plt.tight_layout()

save_figure("global_class_distribution")
plt.show()

## 4 — Fonctions de split, prétraitement, MiniLM et XGBoost

In [ ]:
def make_splits(
    data: pd.DataFrame,
    protocol: str,
    seed: int,
    label_col: str,
    train_fraction: float,
    val_fraction: float,
    cal_fraction: float,
    test_fraction: float,
) -> Dict[str, pd.DataFrame]:
    if not math.isclose(train_fraction + val_fraction + cal_fraction + test_fraction, 1.0, rel_tol=0, abs_tol=1e-9):
        raise ValueError("Les fractions de split doivent sommer à 1.0.")

    if protocol == "iid_stratified":
        train_df, temp_df = train_test_split(
            data,
            test_size=1.0 - train_fraction,
            stratify=data[label_col],
            random_state=seed,
        )
        relative_val = val_fraction / (val_fraction + cal_fraction + test_fraction)
        val_df, remain_df = train_test_split(
            temp_df,
            test_size=1.0 - relative_val,
            stratify=temp_df[label_col],
            random_state=seed + 1,
        )
        relative_cal = cal_fraction / (cal_fraction + test_fraction)
        cal_df, test_df = train_test_split(
            remain_df,
            test_size=1.0 - relative_cal,
            stratify=remain_df[label_col],
            random_state=seed + 2,
        )
        splits = {
            "train": train_df.reset_index(drop=True),
            "val": val_df.reset_index(drop=True),
            "cal": cal_df.reset_index(drop=True),
            "test": test_df.reset_index(drop=True),
        }
        return splits

    if protocol == "temporal_within_file":
        buckets = {"train": [], "val": [], "cal": [], "test": []}
        for source_file, group in data.groupby("__source_file__", sort=False):
            group = group.sort_values("__row_in_file__").reset_index(drop=True)
            n = len(group)

            train_end = int(round(train_fraction * n))
            val_end = train_end + int(round(val_fraction * n))
            cal_end = val_end + int(round(cal_fraction * n))

            # Garantir au moins une ligne quand c'est possible.
            train_end = max(1, min(train_end, n))
            val_end = max(train_end + 1, min(val_end, n)) if n >= 2 else train_end
            cal_end = max(val_end + 1, min(cal_end, n)) if n >= 3 else val_end

            buckets["train"].append(group.iloc[:train_end])
            buckets["val"].append(group.iloc[train_end:val_end])
            buckets["cal"].append(group.iloc[val_end:cal_end])
            buckets["test"].append(group.iloc[cal_end:])

        splits = {
            name: pd.concat(parts, ignore_index=True).reset_index(drop=True)
            for name, parts in buckets.items()
        }
        return splits

    raise ValueError(f"Protocole inconnu : {protocol}")

def capped_indices_per_class(y: np.ndarray, cap: int, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    selected = []
    for cls in np.unique(y):
        indices = np.flatnonzero(y == cls)
        if len(indices) > cap:
            indices = rng.choice(indices, size=cap, replace=False)
        selected.append(indices)
    return np.sort(np.concatenate(selected))

def prepare_split_artifacts(
    splits: Dict[str, pd.DataFrame],
    cfg: Config,
    label_col: str,
) -> Dict[str, object]:
    label_encoder = LabelEncoder()
    label_encoder.fit(splits["train"][label_col])
    class_names = label_encoder.classes_.tolist()
    n_classes = len(class_names)

    all_feature_columns = [
        c for c in splits["train"].columns
        if c not in {label_col, "__source_file__", "__row_in_file__"}
    ]

    X_train_all = splits["train"][all_feature_columns].to_numpy(dtype=np.float64)
    y_train_all = label_encoder.transform(splits["train"][label_col])

    imputer = SimpleImputer(strategy="median")
    X_train_imp = imputer.fit_transform(X_train_all)

    variance_filter = VarianceThreshold(0.0)
    X_train_var = variance_filter.fit_transform(X_train_imp)

    variance_features = [
        feature
        for feature, keep in zip(all_feature_columns, variance_filter.get_support())
        if keep
    ]

    selection_indices = capped_indices_per_class(
        y_train_all,
        cfg.feature_selection_rows_per_class,
        cfg.seeds[0],
    )

    selector = RandomForestClassifier(
        n_estimators=300,
        max_depth=18,
        min_samples_leaf=2,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=cfg.seeds[0],
    )
    selector.fit(X_train_var[selection_indices], y_train_all[selection_indices])

    importance_df = pd.DataFrame({
        "feature": variance_features,
        "importance": selector.feature_importances_,
    }).sort_values("importance", ascending=False).reset_index(drop=True)

    numeric_features = importance_df.head(min(cfg.numeric_top_k, len(importance_df)))["feature"].tolist()

    # Éviter quelques redondances ou doublons directs.
    cleaned_text_features = []
    seen_prefixes = set()
    for feature in numeric_features:
        prefix = feature.replace("1", "")
        if prefix in seen_prefixes:
            continue
        seen_prefixes.add(prefix)
        cleaned_text_features.append(feature)
        if len(cleaned_text_features) >= cfg.text_top_k:
            break
    text_features = cleaned_text_features

    variance_index = {name: idx for idx, name in enumerate(variance_features)}
    numeric_positions = [variance_index[name] for name in numeric_features]
    text_positions = [numeric_features.index(name) for name in text_features]

    scaler = StandardScaler()

    arrays = {}
    for split_name, split_df in splits.items():
        X = split_df[all_feature_columns].to_numpy(dtype=np.float64)
        X_imp = imputer.transform(X)
        X_var = variance_filter.transform(X_imp)
        X_num = X_var[:, numeric_positions].astype(np.float32)
        if split_name == "train":
            X_scaled = scaler.fit_transform(X_num).astype(np.float32)
        else:
            X_scaled = scaler.transform(X_num).astype(np.float32)
        arrays[split_name] = {
            "raw_numeric": X_num,
            "scaled_numeric": X_scaled,
            "scaled_text": X_scaled[:, text_positions],
            "y": label_encoder.transform(split_df[label_col]).astype(np.int64),
        }

    return {
        "label_encoder": label_encoder,
        "class_names": class_names,
        "n_classes": n_classes,
        "all_feature_columns": all_feature_columns,
        "numeric_features": numeric_features,
        "text_features": text_features,
        "importance_df": importance_df,
        "imputer": imputer,
        "variance_filter": variance_filter,
        "scaler": scaler,
        "arrays": arrays,
    }

def compact_prompt(text_features: List[str], values: np.ndarray) -> str:
    return "classify: " + " ".join(
        f"{name}={float(value):.4f}"
        for name, value in zip(text_features, values)
    )

class IndexedDataset(Dataset):
    def __init__(self, labels: np.ndarray, indices: np.ndarray):
        self.labels = labels
        self.indices = np.asarray(indices)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, item: int):
        idx = int(self.indices[item])
        return {"index": idx, "label": int(self.labels[idx])}

class CompactCollator:
    def __init__(self, tokenizer, scaled_text: np.ndarray, text_features: List[str], max_length: int):
        self.tokenizer = tokenizer
        self.scaled_text = scaled_text
        self.text_features = text_features
        self.max_length = max_length

    def __call__(self, items):
        indices = [item["index"] for item in items]
        labels = torch.tensor([item["label"] for item in items], dtype=torch.long)
        texts = [compact_prompt(self.text_features, self.scaled_text[idx]) for idx in indices]
        encoded = self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=self.max_length,
            pad_to_multiple_of=8,
            return_tensors="pt",
        )
        encoded["labels"] = labels
        encoded["indices"] = torch.tensor(indices, dtype=torch.long)
        return encoded



def metrics_from_probs(probabilities: np.ndarray, labels: np.ndarray, n_classes: int) -> Dict[str, float]:
    predictions = probabilities.argmax(axis=1)
    all_labels = np.arange(n_classes)
    present_labels = np.unique(labels)

    macro_f1_present = float(f1_score(labels, predictions, labels=present_labels, average="macro", zero_division=0))
    macro_f1_all = float(f1_score(labels, predictions, labels=all_labels, average="macro", zero_division=0))
    macro_precision_present = float(precision_score(labels, predictions, labels=present_labels, average="macro", zero_division=0))
    macro_precision_all = float(precision_score(labels, predictions, labels=all_labels, average="macro", zero_division=0))
    macro_recall_present = float(recall_score(labels, predictions, labels=present_labels, average="macro", zero_division=0))
    macro_recall_all = float(recall_score(labels, predictions, labels=all_labels, average="macro", zero_division=0))

    return {
        "accuracy": float(accuracy_score(labels, predictions)),
        "macro_f1": macro_f1_present,
        "macro_f1_present": macro_f1_present,
        "macro_f1_all": macro_f1_all,
        "weighted_f1": float(f1_score(labels, predictions, average="weighted", zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(labels, predictions)),
        "mcc": float(matthews_corrcoef(labels, predictions)),
        "macro_precision": macro_precision_present,
        "macro_precision_present": macro_precision_present,
        "macro_precision_all": macro_precision_all,
        "macro_recall": macro_recall_present,
        "macro_recall_present": macro_recall_present,
        "macro_recall_all": macro_recall_all,
        "nll": float(log_loss(labels, probabilities, labels=all_labels)),
        "present_class_count": int(len(present_labels)),
        "label_space_coverage": float(len(present_labels) / max(1, n_classes)),
    }


def dir_size_mb(path: Path) -> float:
    return float(sum(p.stat().st_size for p in Path(path).rglob("*") if p.is_file()) / (1024 ** 2))


def build_class_weights(y_train_small: np.ndarray, n_classes: int, cfg: Config) -> torch.Tensor:
    counts = np.bincount(y_train_small, minlength=n_classes).astype(np.float64)
    if cfg.use_sqrt_class_weights:
        weights = np.sqrt(
            len(y_train_small) / (n_classes * np.maximum(counts, 1.0))
        )
        weights = np.clip(weights, cfg.class_weight_min, cfg.class_weight_max)
        weights = weights / weights.mean()
    else:
        weights = np.ones(n_classes, dtype=np.float64)
    return torch.tensor(weights, dtype=torch.float32)

def build_slm_model(model_checkpoint: str, n_classes: int, class_names: List[str], cfg: Config):
    base_model = AutoModelForSequenceClassification.from_pretrained(
        model_checkpoint,
        num_labels=n_classes,
        id2label={i: name for i, name in enumerate(class_names)},
        label2id={name: i for i, name in enumerate(class_names)},
        ignore_mismatched_sizes=True,
    )
    linear_leaf_names = sorted({
        name.split(".")[-1]
        for name, module in base_model.named_modules()
        if isinstance(module, nn.Linear)
    })
    target_modules = list(cfg.lora_target_modules)
    missing = [name for name in target_modules if name not in linear_leaf_names]
    if missing:
        raise ValueError(f"Modules LoRA introuvables : {missing}")

    lora_config = LoraConfig(
        r=cfg.lora_r,
        lora_alpha=cfg.lora_alpha,
        lora_dropout=cfg.lora_dropout,
        target_modules=target_modules,
        modules_to_save=list(BACKBONE_SPECS[cfg.current_backbone]["modules_to_save"]),
        bias="none",
        task_type=TaskType.SEQ_CLS,
    )
    model = get_peft_model(base_model, lora_config).to(DEVICE)

    n_total = sum(parameter.numel() for parameter in model.parameters())
    n_trainable = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
    n_lora = sum(
        parameter.numel()
        for name, parameter in model.named_parameters()
        if parameter.requires_grad and "lora_" in name
    )
    n_head = n_trainable - n_lora

    stats = {
        "llm_total_parameters": n_total,
        "llm_lora_parameters": n_lora,
        "llm_head_parameters": n_head,
        "llm_trainable_parameters": n_trainable,
        "llm_trainable_percent": 100.0 * n_trainable / n_total,
    }
    return model, stats

@torch.no_grad()
def collect_llm_probs(
    model,
    loader: DataLoader,
    device: torch.device,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    model.eval()
    all_probs, all_labels, all_indices = [], [], []
    for batch in tqdm(loader, leave=False):
        labels = batch.pop("labels")
        indices = batch.pop("indices")
        batch = {key: value.to(device, non_blocking=True) for key, value in batch.items()}
        logits = model(**batch).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
        all_probs.append(probs)
        all_labels.append(labels.numpy())
        all_indices.append(indices.numpy())
    return (
        np.concatenate(all_probs),
        np.concatenate(all_labels),
        np.concatenate(all_indices),
    )

def train_slm(
    arrays: Dict[str, Dict[str, np.ndarray]],
    text_features: List[str],
    class_names: List[str],
    n_classes: int,
    seed: int,
    run_dir: Path,
    cfg: Config,
) -> Dict[str, object]:
    set_seed(seed)

    tokenizer = AutoTokenizer.from_pretrained(cfg.model_checkpoint)

    train_indices = capped_indices_per_class(arrays["train"]["y"], cfg.llm_train_cap_per_class, seed + 10)
    val_indices = capped_indices_per_class(arrays["val"]["y"], cfg.llm_val_cap_per_class, seed + 11)

    class_weights = build_class_weights(arrays["train"]["y"][train_indices], n_classes, cfg)

    train_dataset = IndexedDataset(arrays["train"]["y"], train_indices)
    val_dataset = IndexedDataset(arrays["val"]["y"], val_indices)

    train_collator = CompactCollator(tokenizer, arrays["train"]["scaled_text"], text_features, cfg.max_length)
    val_collator = CompactCollator(tokenizer, arrays["val"]["scaled_text"], text_features, cfg.max_length)

    train_loader = DataLoader(
        train_dataset,
        batch_size=cfg.batch_train,
        shuffle=True,
        num_workers=0,
        pin_memory=(DEVICE.type == "cuda"),
        collate_fn=train_collator,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=cfg.batch_eval,
        shuffle=False,
        num_workers=0,
        pin_memory=(DEVICE.type == "cuda"),
        collate_fn=val_collator,
    )

    model, param_stats = build_slm_model(cfg.model_checkpoint, n_classes, class_names, cfg)
    param_stats["backbone"] = cfg.current_backbone
    param_stats["backbone_display"] = backbone_display_name(cfg.current_backbone)
    param_stats["model_checkpoint"] = cfg.model_checkpoint

    criterion = nn.CrossEntropyLoss(
        weight=class_weights.to(DEVICE),
        label_smoothing=cfg.label_smoothing,
    )
    optimizer = AdamW(
        [parameter for parameter in model.parameters() if parameter.requires_grad],
        lr=cfg.learning_rate,
        weight_decay=cfg.weight_decay,
    )

    steps_per_epoch = math.ceil(len(train_loader) / cfg.grad_accum)
    total_steps = max(1, cfg.epochs * steps_per_epoch)
    warmup_steps = int(cfg.warmup_fraction * total_steps)

    def lr_lambda(step: int) -> float:
        if step < warmup_steps:
            return float(step) / float(max(1, warmup_steps))
        progress = (step - warmup_steps) / float(max(1, total_steps - warmup_steps))
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))

    scheduler = LambdaLR(optimizer, lr_lambda=lr_lambda)
    amp_enabled = DEVICE.type == "cuda"
    grad_scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)

    best_state = None
    best_epoch = -1
    best_macro = -np.inf
    patience = 0
    history = []

    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()

    train_start = time.perf_counter()

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running_loss = 0.0

        progress = tqdm(
            train_loader,
            desc=f"{backbone_display_name(cfg.current_backbone)} seed={seed} epoch {epoch}/{cfg.epochs}",
        )
        for step, batch in enumerate(progress, start=1):
            labels = batch.pop("labels").to(DEVICE, non_blocking=True)
            batch.pop("indices")
            batch = {key: value.to(DEVICE, non_blocking=True) for key, value in batch.items()}

            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=amp_enabled):
                logits = model(**batch).logits
                loss = criterion(logits, labels) / cfg.grad_accum

            grad_scaler.scale(loss).backward()
            running_loss += float(loss.detach().cpu()) * cfg.grad_accum

            if step % cfg.grad_accum == 0 or step == len(train_loader):
                grad_scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(
                    [parameter for parameter in model.parameters() if parameter.requires_grad],
                    max_norm=1.0,
                )
                grad_scaler.step(optimizer)
                grad_scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()

            progress.set_postfix(loss=f"{running_loss/step:.4f}")

        val_probs_small, val_labels_small, _ = collect_llm_probs(model, val_loader, DEVICE)
        val_metrics_small = metrics_from_probs(val_probs_small, val_labels_small, n_classes)
        row = {
            "epoch": epoch,
            "train_loss": running_loss / max(1, len(train_loader)),
            **{f"val_bal_{key}": value for key, value in val_metrics_small.items()},
        }
        history.append(row)
        print(pd.Series(row).round(6))

        if val_metrics_small["macro_f1"] > best_macro:
            best_macro = val_metrics_small["macro_f1"]
            best_epoch = epoch
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            patience = 0
            print(
                f"🏆 Nouveau meilleur {backbone_display_name(cfg.current_backbone)} "
                f"(seed={seed}) : Macro-F1={best_macro:.6f}"
            )
        else:
            patience += 1
            if patience >= cfg.early_stopping_patience:
                print(f"Early stopping {backbone_display_name(cfg.current_backbone)}.")
                break

    if best_state is None:
        raise RuntimeError("Aucun checkpoint MiniLM sauvegardé.")

    model.load_state_dict(best_state, strict=True)

    llm_train_seconds = time.perf_counter() - train_start
    peak_vram_gib = (
        torch.cuda.max_memory_allocated() / 2**30
        if DEVICE.type == "cuda"
        else 0.0
    )

    history_df = pd.DataFrame(history)
    history_df.to_csv(run_dir / "minilm_training_history.csv", index=False)

    return {
        "tokenizer": tokenizer,
        "model": model,
        "history_df": history_df,
        "best_epoch": best_epoch,
        "best_balanced_val_macro_f1": best_macro,
        "train_seconds": llm_train_seconds,
        "peak_vram_gib": peak_vram_gib,
        "param_stats": param_stats,
        "train_indices": train_indices,
        "val_indices": val_indices,
        "class_weights": class_weights.numpy(),
    }

def train_xgboost(
    arrays: Dict[str, Dict[str, np.ndarray]],
    n_classes: int,
    seed: int,
    cfg: Config,
) -> Tuple[object, float]:
    xgb_common = dict(
        objective="multi:softprob",
        num_class=n_classes,
        n_estimators=cfg.xgb_n_estimators,
        max_depth=cfg.xgb_max_depth,
        learning_rate=cfg.xgb_learning_rate,
        subsample=cfg.xgb_subsample,
        colsample_bytree=cfg.xgb_colsample_bytree,
        min_child_weight=cfg.xgb_min_child_weight,
        reg_lambda=cfg.xgb_reg_lambda,
        max_bin=256,
        tree_method="hist",
        eval_metric="mlogloss",
        early_stopping_rounds=cfg.xgb_early_stopping_rounds,
        random_state=seed,
        n_jobs=-1,
    )

    model = XGBClassifier(
        **xgb_common,
        device="cuda" if DEVICE.type == "cuda" else "cpu",
    )

    start = time.perf_counter()
    try:
        model.fit(
            arrays["train"]["raw_numeric"],
            arrays["train"]["y"],
            eval_set=[(arrays["val"]["raw_numeric"], arrays["val"]["y"])],
            verbose=False,
        )
    except Exception as exc:
        print("⚠️ XGBoost GPU indisponible, reprise CPU :", exc)
        model = XGBClassifier(**xgb_common, device="cpu")
        model.fit(
            arrays["train"]["raw_numeric"],
            arrays["train"]["y"],
            eval_set=[(arrays["val"]["raw_numeric"], arrays["val"]["y"])],
            verbose=False,
        )
    seconds = time.perf_counter() - start
    return model, seconds

def selection_score(metrics: Dict[str, float], cfg: Config) -> float:
    if cfg.fusion_selection_metric not in metrics:
        raise KeyError(cfg.fusion_selection_metric)
    return metrics[cfg.fusion_selection_metric]

def make_full_llm_loader(
    arrays: Dict[str, Dict[str, np.ndarray]],
    split_name: str,
    tokenizer,
    text_features: List[str],
    cfg: Config,
) -> DataLoader:
    indices = np.arange(len(arrays[split_name]["y"]))
    dataset = IndexedDataset(arrays[split_name]["y"], indices)
    collator = CompactCollator(tokenizer, arrays[split_name]["scaled_text"], text_features, cfg.max_length)
    return DataLoader(
        dataset,
        batch_size=cfg.batch_eval,
        shuffle=False,
        num_workers=0,
        pin_memory=(DEVICE.type == "cuda"),
        collate_fn=collator,
    )

def align_probabilities(
    probabilities: np.ndarray,
    source_classes: np.ndarray,
    n_classes: int,
) -> np.ndarray:
    """Aligne les colonnes d'un classifieur entraîné sur un sous-ensemble de classes."""
    probabilities = np.asarray(probabilities, dtype=np.float64)
    source_classes = np.asarray(source_classes, dtype=np.int64)
    aligned = np.zeros((len(probabilities), n_classes), dtype=np.float64)
    aligned[:, source_classes] = probabilities
    row_sums = aligned.sum(axis=1, keepdims=True)
    zero_rows = row_sums[:, 0] <= 0
    if np.any(zero_rows):
        aligned[zero_rows] = 1.0 / n_classes
        row_sums = aligned.sum(axis=1, keepdims=True)
    return aligned / row_sums


def split_selection_and_conformal_indices(labels: np.ndarray, seed: int) -> Tuple[np.ndarray, np.ndarray]:
    indices = np.arange(len(labels))
    unique, counts = np.unique(labels, return_counts=True)
    stratify = labels if len(unique) > 1 and counts.min() >= 2 else None
    select_idx, conformal_idx = train_test_split(
        indices,
        test_size=0.50,
        random_state=seed + 31,
        stratify=stratify,
    )
    return np.sort(select_idx), np.sort(conformal_idx)


## 5 — Fonction principale d’expérience

In [ ]:
def run_single_experiment(
    data: pd.DataFrame,
    protocol: str,
    seed: int,
    cfg: Config,
    output_dir: Path,
) -> Dict[str, object]:
    set_seed(seed)

    backbone = cfg.current_backbone
    backbone_name = backbone_display_name(backbone)
    run_name = f"{protocol}__{backbone}__seed_{seed}"
    run_dir = output_dir / run_name
    run_dir.mkdir(parents=True, exist_ok=True)

    print("\n" + "=" * 90)
    print(f"RUN : {run_name} | {backbone_name}")
    print("=" * 90)

    splits = make_splits(
        data=data,
        protocol=protocol,
        seed=seed,
        label_col=LABEL_COL,
        train_fraction=cfg.train_fraction,
        val_fraction=cfg.val_fraction,
        cal_fraction=cfg.cal_fraction,
        test_fraction=cfg.test_fraction,
    )

    split_sizes = []
    for split_name, split_df in splits.items():
        counts = split_df[LABEL_COL].value_counts().rename_axis("label").reset_index(name="count")
        counts["split"] = split_name
        counts["protocol"] = protocol
        counts["backbone"] = backbone
        counts["seed"] = seed
        split_sizes.append(counts)
        print(f"\n{split_name.upper():5s}: {len(split_df):,} lignes")
        print(split_df[LABEL_COL].value_counts().to_string())
    split_distribution_df = pd.concat(split_sizes, ignore_index=True)
    split_distribution_df.to_csv(run_dir / "split_distribution.csv", index=False)

    artifacts = prepare_split_artifacts(splits, cfg, LABEL_COL)
    arrays = artifacts["arrays"]
    class_names = artifacts["class_names"]
    n_classes = artifacts["n_classes"]
    numeric_features = artifacts["numeric_features"]
    text_features = artifacts["text_features"]
    importance_df = artifacts["importance_df"]

    importance_df.to_csv(run_dir / "feature_selection_importance.csv", index=False)

    # Audit de longueur des tokens sur un sous-échantillon train.
    audit_size = min(10_000, len(arrays["train"]["y"]))
    audit_indices = np.arange(audit_size)
    tokenizer_audit = AutoTokenizer.from_pretrained(cfg.model_checkpoint)
    token_lengths = []
    for idx in tqdm(audit_indices, desc=f"Audit tokens {run_name}"):
        prompt = compact_prompt(text_features, arrays["train"]["scaled_text"][idx])
        token_lengths.append(len(tokenizer_audit(prompt, truncation=False, add_special_tokens=True)["input_ids"]))
    token_lengths = np.asarray(token_lengths, dtype=np.int64)
    token_audit = {
        "protocol": protocol,
        "backbone": backbone,
        "backbone_display": backbone_name,
        "model_checkpoint": cfg.model_checkpoint,
        "seed": seed,
        "n_audit": int(audit_size),
        "text_feature_count": int(len(text_features)),
        "max_length": int(cfg.max_length),
        "mean_tokens": float(token_lengths.mean()),
        "median_tokens": float(np.median(token_lengths)),
        "p95_tokens": float(np.percentile(token_lengths, 95)),
        "max_tokens": int(token_lengths.max()),
        "truncation_rate": float(np.mean(token_lengths > cfg.max_length)),
    }
    pd.DataFrame([token_audit]).to_csv(run_dir / "token_audit.csv", index=False)

    # MiniLM
    llm_bundle = train_slm(arrays, text_features, class_names, n_classes, seed, run_dir, cfg)
    llm_model = llm_bundle["model"]
    llm_tokenizer = llm_bundle["tokenizer"]

    # XGBoost
    xgb_model, xgb_train_seconds = train_xgboost(arrays, n_classes, seed, cfg)

    probabilities = {}
    for split_name in ["val", "cal", "test"]:
        llm_loader = make_full_llm_loader(arrays, split_name, llm_tokenizer, text_features, cfg)

        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        start = time.perf_counter()
        llm_probs, labels, indices = collect_llm_probs(llm_model, llm_loader, DEVICE)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        llm_seconds = time.perf_counter() - start

        xgb_probs = xgb_model.predict_proba(arrays[split_name]["raw_numeric"])
        xgb_probs = np.asarray(xgb_probs, dtype=np.float64)
        xgb_probs = xgb_probs / xgb_probs.sum(axis=1, keepdims=True)

        probabilities[split_name] = {
            "llm": llm_probs.astype(np.float64),
            "xgb": xgb_probs,
            "labels": labels,
            "indices": indices,
            "llm_seconds": llm_seconds,
        }

    # CAL est séparé : une moitié pour sélectionner la fusion, l'autre pour la conformité.
    cal_labels_all = probabilities["cal"]["labels"]
    cal_llm_all = probabilities["cal"]["llm"]
    cal_xgb_all = probabilities["cal"]["xgb"]
    cal_select_idx, cal_conformal_idx = split_selection_and_conformal_indices(cal_labels_all, seed)

    cal_labels = cal_labels_all[cal_select_idx]
    cal_llm = cal_llm_all[cal_select_idx]
    cal_xgb = cal_xgb_all[cal_select_idx]

    np.savez(
        run_dir / "calibration_partition_indices.npz",
        selection_indices=cal_select_idx,
        conformal_indices=cal_conformal_idx,
    )

    candidate_cal_metrics: Dict[str, Dict[str, float]] = {}
    candidate_test_probs: Dict[str, np.ndarray] = {}

    candidate_cal_metrics["slm"] = metrics_from_probs(cal_llm, cal_labels, n_classes)
    candidate_test_probs["slm"] = probabilities["test"]["llm"]

    candidate_cal_metrics["xgboost"] = metrics_from_probs(cal_xgb, cal_labels, n_classes)
    candidate_test_probs["xgboost"] = probabilities["test"]["xgb"]

    minimum_alpha = 0.0
    maximum_alpha = 1.0 - cfg.fusion_min_semantic_weight
    alphas = np.linspace(minimum_alpha, maximum_alpha, cfg.fusion_grid_points)

    best_alpha = None
    best_alpha_metrics = None
    best_alpha_score = -np.inf

    for alpha in alphas:
        fused = alpha * cal_xgb + (1.0 - alpha) * cal_llm
        fused = fused / fused.sum(axis=1, keepdims=True)
        metrics = metrics_from_probs(fused, cal_labels, n_classes)
        score = selection_score(metrics, cfg)
        if score > best_alpha_score:
            best_alpha_score = score
            best_alpha = float(alpha)
            best_alpha_metrics = metrics

    candidate_name_fusion = f"fusion_alpha_{best_alpha:.3f}"
    candidate_cal_metrics[candidate_name_fusion] = best_alpha_metrics

    test_alpha_probs = (
        best_alpha * probabilities["test"]["xgb"]
        + (1.0 - best_alpha) * probabilities["test"]["llm"]
    )
    test_alpha_probs /= test_alpha_probs.sum(axis=1, keepdims=True)
    candidate_test_probs[candidate_name_fusion] = test_alpha_probs

    eps = 1e-8
    stack_X_cal = np.concatenate(
        [np.log(cal_xgb + eps), np.log(cal_llm + eps)],
        axis=1,
    )
    stacker = LogisticRegression(
        C=10.0,
        max_iter=1000,
        solver="lbfgs",
        multi_class="multinomial",
        n_jobs=-1,
        random_state=seed,
    )
    stacker.fit(stack_X_cal, cal_labels)
    stack_cal_probs = align_probabilities(
        stacker.predict_proba(stack_X_cal),
        stacker.classes_,
        n_classes,
    )
    candidate_cal_metrics["stacking"] = metrics_from_probs(stack_cal_probs, cal_labels, n_classes)

    stack_X_test = np.concatenate(
        [
            np.log(probabilities["test"]["xgb"] + eps),
            np.log(probabilities["test"]["llm"] + eps),
        ],
        axis=1,
    )
    candidate_test_probs["stacking"] = align_probabilities(
        stacker.predict_proba(stack_X_test),
        stacker.classes_,
        n_classes,
    )

    cal_comparison = pd.DataFrame(candidate_cal_metrics).T.sort_values(
        cfg.fusion_selection_metric,
        ascending=False,
    )
    cal_comparison.to_csv(run_dir / "candidate_calibration_comparison.csv")

    selected_candidate = cal_comparison.index[0]
    final_probs = candidate_test_probs[selected_candidate]
    test_labels = probabilities["test"]["labels"]
    final_predictions = final_probs.argmax(axis=1)
    final_metrics = metrics_from_probs(final_probs, test_labels, n_classes)
    final_report = classification_report(
        test_labels,
        final_predictions,
        labels=np.arange(n_classes),
        target_names=class_names,
        digits=4,
        zero_division=0,
        output_dict=True,
    )
    final_cm = confusion_matrix(test_labels, final_predictions, labels=np.arange(n_classes))

    test_results = {
        name: metrics_from_probs(probs, test_labels, n_classes)
        for name, probs in candidate_test_probs.items()
    }
    test_comparison = pd.DataFrame(test_results).T.sort_values(
        cfg.fusion_selection_metric,
        ascending=False,
    )
    test_comparison.to_csv(run_dir / "candidate_test_comparison.csv")

    branch_summary = pd.DataFrame({
        f"{backbone_name}-LoRA": test_results["slm"],
        "XGBoost": test_results["xgboost"],
        "Final": test_results[selected_candidate],
    }).T
    branch_summary["delta_weighted_f1_vs_minilm"] = (
        branch_summary["weighted_f1"] - branch_summary.loc[f"{backbone_name}-LoRA", "weighted_f1"]
    )
    branch_summary["delta_macro_f1_vs_minilm"] = (
        branch_summary["macro_f1"] - branch_summary.loc[f"{backbone_name}-LoRA", "macro_f1"]
    )
    if "macro_f1_all" in branch_summary.columns:
        branch_summary["delta_macro_f1_all_vs_minilm"] = (
            branch_summary["macro_f1_all"] - branch_summary.loc[f"{backbone_name}-LoRA", "macro_f1_all"]
        )
    branch_summary.to_csv(run_dir / "branch_summary.csv")

    # Sauvegardes des modèles et prédictions.
    adapter_dir = run_dir / "slm_adapter"
    adapter_dir.mkdir(parents=True, exist_ok=True)
    llm_model.save_pretrained(adapter_dir)
    llm_tokenizer.save_pretrained(adapter_dir)
    adapter_size_mb = dir_size_mb(adapter_dir)

    joblib.dump(xgb_model, run_dir / "xgboost_model.joblib")
    joblib.dump(stacker, run_dir / "fusion_stacker.joblib")
    joblib.dump(artifacts["label_encoder"], run_dir / "label_encoder.joblib")
    joblib.dump(artifacts["imputer"], run_dir / "imputer.joblib")
    joblib.dump(artifacts["variance_filter"], run_dir / "variance_filter.joblib")
    joblib.dump(artifacts["scaler"], run_dir / "scaler.joblib")
    joblib.dump(artifacts["all_feature_columns"], run_dir / "input_features.joblib")
    joblib.dump(numeric_features, run_dir / "numeric_features.joblib")
    joblib.dump(text_features, run_dir / "text_features.joblib")

    prediction_df = pd.DataFrame({
        "true_label": [class_names[index] for index in test_labels],
        "predicted_label": [class_names[index] for index in final_predictions],
        "confidence": final_probs.max(axis=1),
    })
    for class_index, class_name in enumerate(class_names):
        prediction_df[f"p_{class_name}"] = final_probs[:, class_index]
    prediction_df.to_csv(run_dir / "test_predictions.csv", index=False)

    with open(run_dir / "classification_report.json", "w", encoding="utf-8") as handle:
        json.dump(final_report, handle, indent=2, ensure_ascii=False)
    np.save(run_dir / "confusion_matrix.npy", final_cm)

    metadata_dict = {
        "protocol": protocol,
        "backbone": backbone,
        "backbone_display": backbone_name,
        "model_checkpoint": cfg.model_checkpoint,
        "seed": seed,
        "selected_candidate": selected_candidate,
        "best_alpha_numeric": best_alpha,
        "lora_target_modules": list(cfg.lora_target_modules),
        "calibration_selection_size": int(len(cal_select_idx)),
        "conformal_calibration_size": int(len(cal_conformal_idx)),
        "text_features": text_features,
        "numeric_features": numeric_features,
        "token_audit": token_audit,
        "param_stats": llm_bundle["param_stats"],
        "llm_train_seconds": llm_bundle["train_seconds"],
        "xgb_train_seconds": xgb_train_seconds,
        "peak_vram_gib": llm_bundle["peak_vram_gib"],
        "adapter_size_mb": adapter_size_mb,
    }
    with open(run_dir / "run_metadata.json", "w", encoding="utf-8") as handle:
        json.dump(metadata_dict, handle, indent=2, ensure_ascii=False)

    # Libération mémoire.
    del llm_model
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    summary_row = {
        "protocol": protocol,
        "backbone": backbone,
        "backbone_display": backbone_name,
        "model_checkpoint": cfg.model_checkpoint,
        "seed": seed,
        "selected_candidate": selected_candidate,
        "best_alpha_numeric": best_alpha,
        "lora_target_modules": list(cfg.lora_target_modules),
        "n_classes": n_classes,
        "classes": " | ".join(class_names),
        "numeric_features": " | ".join(numeric_features),
        "text_features": " | ".join(text_features),
        **llm_bundle["param_stats"],
        "llm_train_seconds": llm_bundle["train_seconds"],
        "xgb_train_seconds": xgb_train_seconds,
        "peak_vram_gib": llm_bundle["peak_vram_gib"],
        "adapter_size_mb": adapter_size_mb,
        **token_audit,
        **{f"final_{key}": value for key, value in final_metrics.items()},
        **{f"slm_test_{key}": value for key, value in test_results["slm"].items()},
        **{f"xgboost_test_{key}": value for key, value in test_results["xgboost"].items()},
    }

    print("\nRÉSULTAT FINAL")
    print(pd.Series(summary_row).head(30))
    print("\nRapport final :")
    print(classification_report(
        test_labels,
        final_predictions,
        labels=np.arange(n_classes),
        target_names=class_names,
        digits=4,
        zero_division=0,
    ))

    return {
        "summary_row": summary_row,
        "split_distribution_df": split_distribution_df,
        "importance_df": importance_df.assign(protocol=protocol, backbone=backbone, seed=seed),
        "token_audit": token_audit,
        "history_df": llm_bundle["history_df"].assign(protocol=protocol, backbone=backbone, seed=seed),
        "branch_summary": branch_summary.assign(protocol=protocol, backbone=backbone, seed=seed),
        "cal_comparison": cal_comparison.assign(protocol=protocol, backbone=backbone, seed=seed),
        "test_comparison": test_comparison.assign(protocol=protocol, backbone=backbone, seed=seed),
        "confusion_matrix": final_cm,
        "class_names": class_names,
        "run_dir": str(run_dir),
        "selected_candidate": selected_candidate,
    }

## 6 — Fonctions de calibration conforme et d’explicabilité v6

In [ ]:

# ================================================================
# TRUSTWORTHINESS v6 : calibration, conformité et explicabilité
# ================================================================

def transform_splits_from_saved_preprocessing(
    splits: Dict[str, pd.DataFrame],
    run_dir: Path,
) -> Dict[str, object]:
    label_encoder = joblib.load(run_dir / "label_encoder.joblib")
    imputer = joblib.load(run_dir / "imputer.joblib")
    variance_filter = joblib.load(run_dir / "variance_filter.joblib")
    scaler = joblib.load(run_dir / "scaler.joblib")
    input_features = joblib.load(run_dir / "input_features.joblib")
    numeric_features = joblib.load(run_dir / "numeric_features.joblib")
    text_features = joblib.load(run_dir / "text_features.joblib")

    variance_features = [
        feature for feature, keep in zip(input_features, variance_filter.get_support()) if keep
    ]
    variance_index = {name: idx for idx, name in enumerate(variance_features)}
    numeric_positions = [variance_index[name] for name in numeric_features]
    text_positions = [numeric_features.index(name) for name in text_features]

    arrays = {}
    for split_name, split_df in splits.items():
        X = split_df[input_features].to_numpy(dtype=np.float64)
        X_imp = imputer.transform(X)
        X_var = variance_filter.transform(X_imp)
        X_num = X_var[:, numeric_positions].astype(np.float32)
        X_scaled = scaler.transform(X_num).astype(np.float32)
        arrays[split_name] = {
            "raw_numeric": X_num,
            "scaled_numeric": X_scaled,
            "scaled_text": X_scaled[:, text_positions],
            "y": label_encoder.transform(split_df[LABEL_COL]).astype(np.int64),
        }

    return {
        "arrays": arrays,
        "label_encoder": label_encoder,
        "class_names": label_encoder.classes_.tolist(),
        "n_classes": len(label_encoder.classes_),
        "numeric_features": numeric_features,
        "text_features": text_features,
    }


def load_slm_adapter(
    run_dir: Path,
    n_classes: int,
    class_names: List[str],
    model_checkpoint: str,
):
    base_model = AutoModelForSequenceClassification.from_pretrained(
        model_checkpoint,
        num_labels=n_classes,
        id2label={i: name for i, name in enumerate(class_names)},
        label2id={name: i for i, name in enumerate(class_names)},
        ignore_mismatched_sizes=True,
    )
    model = PeftModel.from_pretrained(base_model, run_dir / "slm_adapter").to(DEVICE)
    tokenizer = AutoTokenizer.from_pretrained(run_dir / "slm_adapter")
    model.eval()
    return model, tokenizer


def get_candidate_probabilities(
    candidate: str,
    llm_probs: np.ndarray,
    xgb_probs: np.ndarray,
    stacker,
    best_alpha: float,
    n_classes: int,
) -> np.ndarray:
    if candidate == "slm":
        probs = llm_probs
    elif candidate == "xgboost":
        probs = xgb_probs
    elif candidate.startswith("fusion_alpha_"):
        probs = best_alpha * xgb_probs + (1.0 - best_alpha) * llm_probs
    elif candidate == "stacking":
        eps = 1e-8
        features = np.concatenate([np.log(xgb_probs + eps), np.log(llm_probs + eps)], axis=1)
        probs = align_probabilities(stacker.predict_proba(features), stacker.classes_, n_classes)
    else:
        raise ValueError(f"Candidat inconnu : {candidate}")
    probs = np.asarray(probs, dtype=np.float64)
    return probs / probs.sum(axis=1, keepdims=True)


def fit_temperature(probabilities: np.ndarray, labels: np.ndarray) -> float:
    logits = torch.tensor(np.log(np.clip(probabilities, 1e-12, 1.0)), dtype=torch.float64)
    targets = torch.tensor(labels, dtype=torch.long)
    log_temperature = torch.zeros(1, dtype=torch.float64, requires_grad=True)
    optimizer = torch.optim.LBFGS([log_temperature], lr=0.1, max_iter=80, line_search_fn="strong_wolfe")

    def closure():
        optimizer.zero_grad()
        temperature = torch.exp(log_temperature).clamp(0.05, 20.0)
        loss = F.cross_entropy(logits / temperature, targets)
        loss.backward()
        return loss

    optimizer.step(closure)
    return float(torch.exp(log_temperature.detach()).clamp(0.05, 20.0).item())


def apply_temperature(probabilities: np.ndarray, temperature: float) -> np.ndarray:
    logits = np.log(np.clip(probabilities, 1e-12, 1.0)) / temperature
    logits -= logits.max(axis=1, keepdims=True)
    exp_values = np.exp(logits)
    return exp_values / exp_values.sum(axis=1, keepdims=True)


def conformal_true_scores(
    probabilities: np.ndarray,
    labels: np.ndarray,
    method: str,
    raps_lambda: float,
    k_reg: int,
) -> np.ndarray:
    order = np.argsort(-probabilities, axis=1)
    sorted_probs = np.take_along_axis(probabilities, order, axis=1)
    cumulative = np.cumsum(sorted_probs, axis=1)
    ranks = np.empty_like(order)
    ranks[np.arange(len(order))[:, None], order] = np.arange(probabilities.shape[1])[None, :]
    true_rank = ranks[np.arange(len(labels)), labels]
    scores = cumulative[np.arange(len(labels)), true_rank]
    if method == "RAPS":
        scores = scores + raps_lambda * np.maximum(true_rank + 1 - k_reg, 0)
    return scores


def conformal_quantile(scores: np.ndarray, alpha: float) -> float:
    n = len(scores)
    level = min(1.0, math.ceil((n + 1) * (1.0 - alpha)) / n)
    return float(np.quantile(scores, level, method="higher"))


def build_prediction_sets(
    probabilities: np.ndarray,
    qhat: float,
    method: str,
    raps_lambda: float,
    k_reg: int,
) -> List[List[int]]:
    order = np.argsort(-probabilities, axis=1)
    sorted_probs = np.take_along_axis(probabilities, order, axis=1)
    cumulative = np.cumsum(sorted_probs, axis=1)
    rank_numbers = np.arange(1, probabilities.shape[1] + 1)
    scores = cumulative.copy()
    if method == "RAPS":
        scores = scores + raps_lambda * np.maximum(rank_numbers - k_reg, 0)[None, :]

    sets = []
    for i in range(len(probabilities)):
        include = scores[i] <= qhat
        include[0] = True
        sets.append(order[i][include].tolist())
    return sets


def evaluate_prediction_sets(
    prediction_sets: List[List[int]],
    labels: np.ndarray,
    class_names: List[str],
    abstain_size: int,
) -> Tuple[Dict[str, float], pd.DataFrame]:
    sizes = np.asarray([len(values) for values in prediction_sets], dtype=np.int64)
    covered = np.asarray([int(label in values) for label, values in zip(labels, prediction_sets)])
    overall = {
        "coverage": float(covered.mean()),
        "average_set_size": float(sizes.mean()),
        "median_set_size": float(np.median(sizes)),
        "singleton_rate": float(np.mean(sizes == 1)),
        "ambiguity_rate": float(np.mean((sizes > 1) & (sizes <= abstain_size))),
        "abstention_rate": float(np.mean(sizes > abstain_size)),
    }
    rows = []
    for class_index, class_name in enumerate(class_names):
        mask = labels == class_index
        rows.append({
            "class": class_name,
            "support": int(mask.sum()),
            "coverage": float(covered[mask].mean()) if mask.any() else np.nan,
            "average_set_size": float(sizes[mask].mean()) if mask.any() else np.nan,
        })
    return overall, pd.DataFrame(rows)


def stratified_explanation_indices(labels: np.ndarray, n_samples: int, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    present_classes = np.unique(labels)
    per_class = max(1, n_samples // max(1, len(present_classes)))
    selected = []
    for class_index in present_classes:
        indices = np.flatnonzero(labels == class_index)
        take = min(per_class, len(indices))
        selected.extend(rng.choice(indices, size=take, replace=False).tolist())
    remaining = n_samples - len(selected)
    if remaining > 0:
        pool = np.setdiff1d(np.arange(len(labels)), np.asarray(selected, dtype=np.int64))
        if len(pool):
            selected.extend(rng.choice(pool, size=min(remaining, len(pool)), replace=False).tolist())
    return np.asarray(sorted(selected[:n_samples]), dtype=np.int64)


def get_tree_shap_values(xgb_model, X: np.ndarray, n_classes: int) -> np.ndarray:
    raw = xgb_model.get_booster().predict(
        xgb.DMatrix(X),
        pred_contribs=True,
        strict_shape=True,
    )
    raw = np.asarray(raw)
    if raw.ndim == 3:
        return raw[:, :, :-1]
    if raw.ndim == 2 and n_classes > 1:
        return raw.reshape(len(X), n_classes, -1)[:, :, :-1]
    raise RuntimeError(f"Forme TreeSHAP inattendue : {raw.shape}")


def slm_predict_values(model, tokenizer, text_features: List[str], values_batch: np.ndarray) -> np.ndarray:
    prompts = [compact_prompt(text_features, values) for values in values_batch]
    encoded = tokenizer(
        prompts,
        padding=True,
        truncation=True,
        max_length=CFG.max_length,
        return_tensors="pt",
    )
    encoded = {key: value.to(DEVICE) for key, value in encoded.items()}
    with torch.no_grad():
        logits = model(**encoded).logits
    return torch.softmax(logits, dim=-1).cpu().numpy()


def deterministic_template_explanation(
    predicted_label: str,
    confidence: float,
    conformal_labels: List[str],
    shap_evidence: List[Tuple[str, float]],
    occlusion_evidence: List[Tuple[str, float]],
) -> str:
    numeric_text = ", ".join(f"[{name}] ({value:+.3f})" for name, value in shap_evidence)
    semantic_text = ", ".join(f"[{name}] (drop={value:.3f})" for name, value in occlusion_evidence)
    uncertainty = (
        f"The conformal set contains only {predicted_label}."
        if len(conformal_labels) == 1
        else f"The decision is uncertain among: {', '.join(conformal_labels)}."
    )
    return (
        f"The predicted class is {predicted_label} with confidence {confidence:.3f}. "
        f"The strongest numeric evidence is {numeric_text}. "
        f"The SLM evidence is {semantic_text}. {uncertainty}"
    )


_EXPLAINER_CACHE = {}

def get_explanation_slm():
    checkpoint = CFG.explanation_model_checkpoint
    if checkpoint not in _EXPLAINER_CACHE:
        tokenizer = AutoTokenizer.from_pretrained(checkpoint)
        model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint).to(DEVICE)
        model.eval()
        _EXPLAINER_CACHE[checkpoint] = (model, tokenizer)
    return _EXPLAINER_CACHE[checkpoint]


def generate_slm_explanations(prompts: List[str]) -> Tuple[List[str], float]:
    model, tokenizer = get_explanation_slm()
    start = time.perf_counter()
    outputs = []
    batch_size = 8
    for start_index in range(0, len(prompts), batch_size):
        batch_prompts = prompts[start_index:start_index + batch_size]
        encoded = tokenizer(
            batch_prompts,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt",
        )
        encoded = {key: value.to(DEVICE) for key, value in encoded.items()}
        with torch.no_grad():
            generated = model.generate(
                **encoded,
                max_new_tokens=CFG.explanation_max_new_tokens,
                do_sample=False,
                num_beams=2,
                repetition_penalty=1.1,
            )
        outputs.extend(tokenizer.batch_decode(generated, skip_special_tokens=True))
    seconds = time.perf_counter() - start
    return outputs, seconds


def citation_metrics(text: str, allowed_features: List[str]) -> Dict[str, float]:
    citations = re.findall(r"\[([a-zA-Z0-9_]+)\]", text)
    allowed = set(allowed_features)
    valid = [citation for citation in citations if citation in allowed]
    unsupported = [citation for citation in citations if citation not in allowed]
    return {
        "citation_count": len(citations),
        "valid_citation_rate": len(valid) / max(1, len(citations)),
        "unsupported_citation_rate": len(unsupported) / max(1, len(citations)),
        "evidence_coverage": len(set(valid)) / max(1, len(allowed)),
    }


## 6 — Reprise automatique des runs sauvegardés

La cellule suivante vérifie les 12 dossiers attendus dans Google Drive.

Un run est considéré comme terminé uniquement si ses modèles, métadonnées,
prédictions, métriques et matrice de confusion sont tous présents.

- `resume_completed_runs=True` : recharge les runs terminés.
- `force_rerun=False` : interdit leur réentraînement.
- Un run interrompu avant sa sauvegarde finale est le seul run relancé.

In [ ]:
def expected_run_dir(
    output_dir: Path,
    protocol: str,
    backbone: str,
    seed: int,
) -> Path:
    return output_dir / f"{protocol}__{backbone}__seed_{seed}"


def completed_run_files(run_dir: Path) -> List[Path]:
    return [
        run_dir / "run_metadata.json",
        run_dir / "candidate_test_comparison.csv",
        run_dir / "candidate_calibration_comparison.csv",
        run_dir / "branch_summary.csv",
        run_dir / "feature_selection_importance.csv",
        run_dir / "split_distribution.csv",
        run_dir / "token_audit.csv",
        run_dir / "minilm_training_history.csv",
        run_dir / "confusion_matrix.npy",
        run_dir / "classification_report.json",
        run_dir / "label_encoder.joblib",
        run_dir / "test_predictions.csv",
        run_dir / "slm_adapter",
        run_dir / "xgboost_model.joblib",
        run_dir / "fusion_stacker.joblib",
    ]


def is_run_complete(run_dir: Path) -> bool:
    if not run_dir.exists():
        return False

    for path in completed_run_files(run_dir):
        if not path.exists():
            return False
        if path.is_file() and path.stat().st_size == 0:
            return False

    try:
        with open(
            run_dir / "run_metadata.json",
            "r",
            encoding="utf-8",
        ) as handle:
            metadata_dict = json.load(handle)

        selected = metadata_dict["selected_candidate"]
        test_table = pd.read_csv(
            run_dir / "candidate_test_comparison.csv",
            index_col=0,
        )
        if selected not in test_table.index:
            return False

        predictions = pd.read_csv(
            run_dir / "test_predictions.csv",
            nrows=5,
        )
        if predictions.empty:
            return False

    except Exception as exc:
        print(f"Run incomplet ou illisible {run_dir.name}: {exc}")
        return False

    return True


def load_completed_run(run_dir: Path) -> Dict[str, object]:
    """Reconstruit le résultat d'un run sans recharger ses modèles."""
    with open(
        run_dir / "run_metadata.json",
        "r",
        encoding="utf-8",
    ) as handle:
        metadata_dict = json.load(handle)

    protocol = str(metadata_dict["protocol"])
    backbone = str(metadata_dict["backbone"])
    backbone_name = str(metadata_dict["backbone_display"])
    seed = int(metadata_dict["seed"])
    selected_candidate = str(metadata_dict["selected_candidate"])

    label_encoder = joblib.load(run_dir / "label_encoder.joblib")
    class_names = label_encoder.classes_.tolist()
    n_classes = len(class_names)

    test_comparison = pd.read_csv(
        run_dir / "candidate_test_comparison.csv",
        index_col=0,
    )
    cal_comparison = pd.read_csv(
        run_dir / "candidate_calibration_comparison.csv",
        index_col=0,
    )
    branch_summary = pd.read_csv(
        run_dir / "branch_summary.csv",
        index_col=0,
    )
    importance_df = pd.read_csv(
        run_dir / "feature_selection_importance.csv",
    )
    split_distribution_df = pd.read_csv(
        run_dir / "split_distribution.csv",
    )
    history_df = pd.read_csv(
        run_dir / "minilm_training_history.csv",
    )
    token_audit_df = pd.read_csv(
        run_dir / "token_audit.csv",
    )

    if selected_candidate not in test_comparison.index:
        raise KeyError(
            f"Candidat {selected_candidate} absent de "
            f"{run_dir / 'candidate_test_comparison.csv'}"
        )

    final_metrics = test_comparison.loc[selected_candidate].to_dict()
    slm_metrics = test_comparison.loc["slm"].to_dict()
    xgb_metrics = test_comparison.loc["xgboost"].to_dict()

    param_stats = dict(metadata_dict.get("param_stats", {}))
    token_audit = dict(metadata_dict.get("token_audit", {}))

    summary_row = {
        "protocol": protocol,
        "backbone": backbone,
        "backbone_display": backbone_name,
        "model_checkpoint": metadata_dict["model_checkpoint"],
        "seed": seed,
        "selected_candidate": selected_candidate,
        "best_alpha_numeric": float(
            metadata_dict["best_alpha_numeric"]
        ),
        "lora_target_modules": metadata_dict.get(
            "lora_target_modules",
            BACKBONE_SPECS[backbone]["target_modules"],
        ),
        "n_classes": n_classes,
        "classes": " | ".join(class_names),
        "numeric_features": " | ".join(
            metadata_dict["numeric_features"]
        ),
        "text_features": " | ".join(
            metadata_dict["text_features"]
        ),
        **param_stats,
        "llm_train_seconds": float(
            metadata_dict["llm_train_seconds"]
        ),
        "xgb_train_seconds": float(
            metadata_dict["xgb_train_seconds"]
        ),
        "peak_vram_gib": float(
            metadata_dict["peak_vram_gib"]
        ),
        "adapter_size_mb": float(
            metadata_dict.get(
                "adapter_size_mb",
                dir_size_mb(run_dir / "slm_adapter"),
            )
        ),
        **token_audit,
        **{
            f"final_{key}": value
            for key, value in final_metrics.items()
        },
        **{
            f"slm_test_{key}": value
            for key, value in slm_metrics.items()
        },
        **{
            f"xgboost_test_{key}": value
            for key, value in xgb_metrics.items()
        },
    }

    # Ensure context columns exist exactly once.
    for frame in [
        history_df,
        importance_df,
        branch_summary,
        cal_comparison,
        test_comparison,
    ]:
        frame["protocol"] = protocol
        frame["backbone"] = backbone
        frame["seed"] = seed

    if "protocol" not in split_distribution_df.columns:
        split_distribution_df["protocol"] = protocol
    if "backbone" not in split_distribution_df.columns:
        split_distribution_df["backbone"] = backbone
    if "seed" not in split_distribution_df.columns:
        split_distribution_df["seed"] = seed

    if token_audit_df.empty:
        token_audit_record = token_audit
    else:
        token_audit_record = token_audit_df.iloc[0].to_dict()
    token_audit_record["protocol"] = protocol
    token_audit_record["backbone"] = backbone
    token_audit_record["seed"] = seed

    cm = np.load(run_dir / "confusion_matrix.npy")

    print(f"✅ Rechargé sans entraînement : {run_dir.name}")

    return {
        "summary_row": summary_row,
        "split_distribution_df": split_distribution_df,
        "importance_df": importance_df,
        "token_audit": token_audit_record,
        "history_df": history_df,
        "branch_summary": branch_summary,
        "cal_comparison": cal_comparison,
        "test_comparison": test_comparison,
        "confusion_matrix": cm,
        "class_names": class_names,
        "run_dir": str(run_dir),
        "selected_candidate": selected_candidate,
        "loaded_from_disk": True,
    }


def campaign_status_table(
    output_dir: Path,
    cfg: Config,
) -> pd.DataFrame:
    rows = []
    for backbone in cfg.backbones:
        for protocol in cfg.protocols:
            for seed in cfg.seeds:
                run_dir = expected_run_dir(
                    output_dir,
                    protocol,
                    backbone,
                    seed,
                )
                rows.append({
                    "backbone": backbone,
                    "protocol": protocol,
                    "seed": seed,
                    "complete": is_run_complete(run_dir),
                    "run_dir": str(run_dir),
                })
    return pd.DataFrame(rows)

## 7 — Exécution complète des expériences

In [ ]:
all_summaries = []
all_histories = []
all_importances = []
all_branch_summaries = []
all_split_distributions = []
all_cal_comparisons = []
all_test_comparisons = []
all_token_audits = []
confusion_registry = {}

status_before = campaign_status_table(MODEL_DIR, CFG)
display(status_before)

print(
    f"Runs déjà terminés : "
    f"{int(status_before['complete'].sum())}/"
    f"{len(status_before)}"
)

for backbone in CFG.backbones:
    spec = apply_backbone_config(backbone)
    print("\n" + "#" * 100)
    print(
        f"BACKBONE : {spec['display_name']} | "
        f"checkpoint={spec['checkpoint']} | "
        f"LoRA={spec['target_modules']}"
    )
    print("#" * 100)

    for protocol in CFG.protocols:
        for seed in CFG.seeds:
            run_dir = expected_run_dir(
                MODEL_DIR,
                protocol,
                backbone,
                seed,
            )

            should_load = (
                CFG.resume_completed_runs
                and not CFG.force_rerun
                and is_run_complete(run_dir)
            )

            if should_load:
                result = load_completed_run(run_dir)
            else:
                print(
                    f"▶️ Exécution du run manquant : "
                    f"{protocol} | {backbone} | seed={seed}"
                )
                result = run_single_experiment(
                    data=df,
                    protocol=protocol,
                    seed=seed,
                    cfg=CFG,
                    output_dir=MODEL_DIR,
                )

            all_summaries.append(result["summary_row"])
            all_histories.append(result["history_df"])
            all_importances.append(result["importance_df"])
            all_branch_summaries.append(result["branch_summary"])
            all_split_distributions.append(
                result["split_distribution_df"]
            )
            all_cal_comparisons.append(result["cal_comparison"])
            all_test_comparisons.append(result["test_comparison"])
            all_token_audits.append(result["token_audit"])

            confusion_registry[(protocol, backbone, seed)] = {
                "cm": result["confusion_matrix"],
                "class_names": result["class_names"],
                "selected_candidate": result["selected_candidate"],
                "run_dir": result["run_dir"],
            }

summaries_df = pd.DataFrame(all_summaries)
histories_df = pd.concat(
    all_histories,
    ignore_index=True,
)
feature_importances_df = pd.concat(
    all_importances,
    ignore_index=True,
)
branch_summaries_df = pd.concat(
    all_branch_summaries,
    ignore_index=False,
)
split_distributions_df = pd.concat(
    all_split_distributions,
    ignore_index=True,
)
cal_comparisons_df = pd.concat(
    all_cal_comparisons,
    ignore_index=False,
)
test_comparisons_df = pd.concat(
    all_test_comparisons,
    ignore_index=False,
)
token_audits_df = pd.DataFrame(all_token_audits)

summaries_df.to_csv(
    TABLE_DIR / "all_run_summaries.csv",
    index=False,
)
histories_df.to_csv(
    TABLE_DIR / "all_training_histories.csv",
    index=False,
)
feature_importances_df.to_csv(
    TABLE_DIR / "all_feature_importances.csv",
    index=False,
)
branch_summaries_df.to_csv(
    TABLE_DIR / "all_branch_summaries.csv",
)
split_distributions_df.to_csv(
    TABLE_DIR / "all_split_distributions.csv",
    index=False,
)
cal_comparisons_df.to_csv(
    TABLE_DIR / "all_calibration_comparisons.csv",
)
test_comparisons_df.to_csv(
    TABLE_DIR / "all_test_comparisons.csv",
)
token_audits_df.to_csv(
    TABLE_DIR / "all_token_audits.csv",
    index=False,
)

status_after = campaign_status_table(MODEL_DIR, CFG)
display(status_after)

print(
    f"Expériences disponibles : "
    f"{len(summaries_df)}/{len(status_after)}"
)
print(
    f"Runs complets sur disque : "
    f"{int(status_after['complete'].sum())}/"
    f"{len(status_after)}"
)

if len(summaries_df) != len(status_after):
    raise RuntimeError(
        "La campagne ne contient pas encore tous les runs."
    )

display(summaries_df.head())

## 8 — Tableaux agrégés (moyenne ± écart-type)

In [ ]:
summary_metrics = [
    "final_accuracy",
    "final_macro_f1_present",
    "final_macro_f1_all",
    "final_weighted_f1",
    "final_balanced_accuracy",
    "final_mcc",
    "final_label_space_coverage",
    "llm_train_seconds",
    "xgb_train_seconds",
    "peak_vram_gib",
    "adapter_size_mb",
    "truncation_rate",
]

aggregate_protocol_backbone = (
    summaries_df
    .groupby(["protocol", "backbone_display"])[summary_metrics]
    .agg(["mean", "std"])
)

branch_metrics = [
    "accuracy",
    "macro_f1",
    "macro_f1_all",
    "weighted_f1",
    "balanced_accuracy",
    "mcc",
    "label_space_coverage",
]
aggregate_branch = (
    branch_summaries_df
    .reset_index()
    .rename(columns={"index": "branch"})
    .groupby(["protocol", "backbone", "branch"])[branch_metrics]
    .agg(["mean", "std"])
)

slm_efficiency_table = summaries_df[[
    "protocol",
    "backbone",
    "backbone_display",
    "seed",
    "llm_total_parameters",
    "llm_trainable_parameters",
    "llm_trainable_percent",
    "adapter_size_mb",
    "llm_train_seconds",
    "peak_vram_gib",
    "mean_tokens",
    "p95_tokens",
    "truncation_rate",
]].copy()

aggregate_protocol_backbone.to_csv(
    TABLE_DIR / "aggregate_by_protocol_and_backbone.csv"
)
aggregate_branch.to_csv(
    TABLE_DIR / "aggregate_by_branch_protocol_backbone.csv"
)
slm_efficiency_table.to_csv(
    TABLE_DIR / "slm_efficiency_per_run.csv",
    index=False,
)

display(aggregate_protocol_backbone)
display(aggregate_branch)
display(slm_efficiency_table.head())

## 9 — Graphes : performances finales par protocole

In [ ]:
plot_df = (
    branch_summaries_df
    .reset_index()
    .rename(columns={"index": "branch"})
)
plot_df["scenario"] = (
    plot_df["protocol"].astype(str)
    + "\n"
    + plot_df["backbone"].map(backbone_display_name)
)

metrics_to_plot = [
    ("weighted_f1", "Weighted-F1"),
    ("macro_f1", "Macro-F1 (classes présentes)"),
    ("macro_f1_all", "Macro-F1 (10 classes)"),
    ("mcc", "MCC"),
    ("balanced_accuracy", "Balanced Accuracy"),
]

for metric_key, metric_label in metrics_to_plot:
    grouped = (
        plot_df.groupby(["scenario", "branch"])[metric_key]
        .agg(["mean", "std"])
        .reset_index()
    )

    scenarios = grouped["scenario"].unique().tolist()
    branches = grouped["branch"].unique().tolist()
    x = np.arange(len(scenarios))
    width = 0.8 / max(1, len(branches))

    plt.figure(figsize=(15, 6))
    for i, branch in enumerate(branches):
        subset = (
            grouped[grouped["branch"] == branch]
            .set_index("scenario")
            .reindex(scenarios)
        )
        plt.bar(
            x + (i - (len(branches) - 1) / 2) * width,
            subset["mean"].to_numpy(),
            width=width,
            yerr=subset["std"].fillna(0.0).to_numpy(),
            capsize=4,
            label=branch,
        )

    plt.xticks(x, scenarios)
    plt.ylabel(metric_label)
    plt.ylim(0.0, 1.02)
    plt.title(f"{metric_label} — comparaison MiniLM vs ModernBERT")
    plt.legend(ncol=2)
    save_figure(f"comparative_{metric_key}_by_protocol_backbone")
    plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

# ============================================================
# Préparation des données
# ============================================================

plot_df = (
    branch_summaries_df
    .reset_index()
    .rename(columns={"index": "branch"})
    .copy()
)

plot_df["scenario"] = (
    plot_df["protocol"].astype(str)
    + "\n"
    + plot_df["backbone"].map(backbone_display_name)
)

metrics_to_plot = [
    ("weighted_f1", "Weighted-F1"),
    ("macro_f1", "Macro-F1 — classes présentes"),
    ("macro_f1_all", "Macro-F1 — 10 classes"),
    ("mcc", "Matthews Correlation Coefficient"),
    ("balanced_accuracy", "Balanced Accuracy"),
]

# Ordre stable des scénarios
scenario_order = (
    plot_df[
        ["protocol", "backbone", "scenario"]
    ]
    .drop_duplicates()
    .sort_values(["protocol", "backbone"])
    ["scenario"]
    .tolist()
)

# Ordre stable des branches
branch_order = (
    plot_df["branch"]
    .drop_duplicates()
    .astype(str)
    .tolist()
)

# Noms plus lisibles dans la légende
branch_display_names = {
    branch: str(branch).replace("_", " ").title()
    for branch in branch_order
}

# Palette commune pour toutes les figures
cmap = plt.get_cmap("Set2")
branch_colors = {
    branch: cmap(i % cmap.N)
    for i, branch in enumerate(branch_order)
}

# Motifs utiles pour impression en niveaux de gris
hatch_patterns = ["", "//", "\\\\", "xx", "..", "++"]
branch_hatches = {
    branch: hatch_patterns[i % len(hatch_patterns)]
    for i, branch in enumerate(branch_order)
}


# ============================================================
# Fonction de création d'une figure comparative
# ============================================================

def plot_comparative_metric(
    data,
    metric_key,
    metric_label,
    scenarios,
    branches,
):
    grouped = (
        data
        .groupby(["scenario", "branch"], observed=True)[metric_key]
        .agg(mean="mean", std="std", n="count")
        .reset_index()
    )

    x = np.arange(len(scenarios))
    n_branches = max(1, len(branches))

    # Largeur totale occupée par les barres d'un scénario
    total_group_width = 0.82
    bar_width = total_group_width / n_branches

    fig_width = max(12, 2.0 * len(scenarios))
    fig, ax = plt.subplots(figsize=(fig_width, 7))

    all_means = []

    for i, branch in enumerate(branches):
        subset = (
            grouped[grouped["branch"].astype(str) == str(branch)]
            .set_index("scenario")
            .reindex(scenarios)
        )

        means = subset["mean"].to_numpy(dtype=float)
        stds = subset["std"].fillna(0.0).to_numpy(dtype=float)

        all_means.extend(means[np.isfinite(means)])

        positions = (
            x
            + (i - (n_branches - 1) / 2) * bar_width
        )

        bars = ax.bar(
            positions,
            means,
            width=bar_width * 0.90,
            yerr=stds,
            capsize=4,
            color=branch_colors[branch],
            edgecolor="black",
            linewidth=0.65,
            hatch=branch_hatches[branch],
            error_kw={
                "elinewidth": 1.0,
                "capthick": 1.0,
                "alpha": 0.80,
            },
            label=branch_display_names[branch],
            zorder=3,
        )

        # Ajouter les valeurs au-dessus des barres
        for bar, mean_value, std_value in zip(bars, means, stds):
            if not np.isfinite(mean_value):
                continue

            annotation_y = mean_value + std_value + 0.012

            ax.text(
                bar.get_x() + bar.get_width() / 2,
                annotation_y,
                f"{mean_value:.3f}",
                ha="center",
                va="bottom",
                rotation=90 if len(scenarios) > 5 else 0,
                fontsize=8,
                fontweight="medium",
                clip_on=False,
            )

    # ========================================================
    # Mise en forme
    # ========================================================

    ax.set_xticks(x)
    ax.set_xticklabels(
        scenarios,
        fontsize=10,
        fontweight="medium",
        linespacing=1.25,
    )

    ax.set_ylabel(
        metric_label,
        fontsize=12,
        fontweight="semibold",
        labelpad=10,
    )

    ax.set_xlabel(
        "Protocole expérimental et backbone",
        fontsize=11,
        labelpad=12,
    )

    ax.set_title(
        f"{metric_label} par protocole, backbone et branche",
        fontsize=15,
        fontweight="bold",
        pad=18,
    )

    ax.text(
        0.5,
        1.01,
        "Comparaison des performances de MiniLM et ModernBERT",
        transform=ax.transAxes,
        ha="center",
        va="bottom",
        fontsize=10,
        alpha=0.75,
    )

    # Les métriques sont normalement comprises entre 0 et 1
    maximum_observed = max(all_means) if all_means else 1.0
    upper_limit = min(
        1.12,
        max(1.02, maximum_observed + 0.10)
    )

    ax.set_ylim(0.0, upper_limit)

    # Grille légère
    ax.grid(
        axis="y",
        linestyle="--",
        linewidth=0.75,
        alpha=0.30,
        zorder=0,
    )

    ax.set_axisbelow(True)

    # Ligne de référence facultative à 0.5
    ax.axhline(
        0.5,
        linestyle=":",
        linewidth=1.0,
        alpha=0.35,
        zorder=1,
    )

    # Retirer les bordures supérieures et droites
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.spines["left"].set_alpha(0.45)
    ax.spines["bottom"].set_alpha(0.45)

    ax.tick_params(
        axis="x",
        length=0,
        pad=8,
    )

    ax.tick_params(
        axis="y",
        labelsize=9,
    )

    # Légende sous le graphique
    legend = ax.legend(
        title="Branche expérimentale",
        loc="upper center",
        bbox_to_anchor=(0.5, -0.18),
        ncol=min(4, len(branches)),
        frameon=False,
        fontsize=9,
        title_fontsize=10,
    )

    plt.tight_layout(rect=[0, 0.07, 1, 1])

    save_figure(
        f"comparative_{metric_key}_by_protocol_backbone"
    )

    plt.show()
    plt.close(fig)


# ============================================================
# Génération des figures
# ============================================================

for metric_key, metric_label in metrics_to_plot:
    plot_comparative_metric(
        data=plot_df,
        metric_key=metric_key,
        metric_label=metric_label,
        scenarios=scenario_order,
        branches=branch_order,
    )

## 10 — Graphe : comparaison frugalité / performance

In [ ]:
eff_df = summaries_df.copy()
eff_df["weighted_f1_pct"] = 100.0 * eff_df["final_weighted_f1"]
eff_df["train_minutes"] = eff_df["llm_train_seconds"] / 60.0

plt.figure(figsize=(11, 7))
for (protocol, backbone), subset in eff_df.groupby(["protocol", "backbone"]):
    plt.scatter(
        subset["train_minutes"],
        subset["weighted_f1_pct"],
        s=120,
        label=f"{backbone_display_name(backbone)} — {protocol}",
    )
    for _, row in subset.iterrows():
        plt.annotate(
            f"seed={int(row['seed'])}",
            (row["train_minutes"], row["weighted_f1_pct"]),
            xytext=(4, 4),
            textcoords="offset points",
            fontsize=8,
        )

plt.xlabel("Temps d'entraînement du SLM (minutes)")
plt.ylabel("Weighted-F1 final (%)")
plt.title("Efficacité : performance vs temps d'entraînement")
plt.legend(fontsize=9)
save_figure("comparative_efficiency_vs_weighted_f1")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

# ============================================================
# Efficacité : performance vs temps d'entraînement
# ============================================================

eff_df = summaries_df.copy()

eff_df["weighted_f1_pct"] = (
    100.0 * eff_df["final_weighted_f1"]
)

eff_df["train_minutes"] = (
    eff_df["llm_train_seconds"] / 60.0
)

# Retirer les observations incomplètes
eff_df = eff_df.dropna(
    subset=[
        "protocol",
        "backbone",
        "seed",
        "weighted_f1_pct",
        "train_minutes",
    ]
).copy()

# Ordre stable des configurations
grouped_efficiency = list(
    eff_df.groupby(
        ["protocol", "backbone"],
        sort=True,
        observed=True,
    )
)

# Palette et marqueurs cohérents
cmap = plt.get_cmap("tab10")

marker_styles = [
    "o",
    "s",
    "^",
    "D",
    "P",
    "X",
    "v",
    "<",
    ">",
]

fig, ax = plt.subplots(figsize=(12, 7.5))

for group_index, ((protocol, backbone), subset) in enumerate(
    grouped_efficiency
):
    subset = (
        subset
        .sort_values("seed")
        .reset_index(drop=True)
    )

    color = cmap(group_index % cmap.N)
    marker = marker_styles[
        group_index % len(marker_styles)
    ]

    display_name = (
        f"{backbone_display_name(backbone)} — {protocol}"
    )

    # --------------------------------------------------------
    # Exécutions individuelles
    # --------------------------------------------------------

    ax.scatter(
        subset["train_minutes"],
        subset["weighted_f1_pct"],
        s=105,
        marker=marker,
        color=color,
        edgecolor="white",
        linewidth=1.0,
        alpha=0.78,
        label=display_name,
        zorder=3,
    )

    # Annotations des seeds avec décalage alterné
    annotation_offsets = [
        (6, 7),
        (6, -13),
        (-38, 7),
        (-38, -13),
    ]

    for row_index, row in subset.iterrows():
        offset_x, offset_y = annotation_offsets[
            row_index % len(annotation_offsets)
        ]

        ax.annotate(
            f"seed={int(row['seed'])}",
            xy=(
                row["train_minutes"],
                row["weighted_f1_pct"],
            ),
            xytext=(offset_x, offset_y),
            textcoords="offset points",
            fontsize=8,
            color=color,
            alpha=0.90,
            arrowprops={
                "arrowstyle": "-",
                "linewidth": 0.55,
                "color": color,
                "alpha": 0.50,
            },
            zorder=4,
        )

    # --------------------------------------------------------
    # Moyenne et dispersion de la configuration
    # --------------------------------------------------------

    mean_time = subset["train_minutes"].mean()
    mean_f1 = subset["weighted_f1_pct"].mean()

    std_time = subset["train_minutes"].std(ddof=1)
    std_f1 = subset["weighted_f1_pct"].std(ddof=1)

    # Un seul seed produit un écart-type NaN
    std_time = 0.0 if np.isnan(std_time) else std_time
    std_f1 = 0.0 if np.isnan(std_f1) else std_f1

    ax.errorbar(
        mean_time,
        mean_f1,
        xerr=std_time,
        yerr=std_f1,
        fmt="none",
        ecolor=color,
        elinewidth=1.4,
        capsize=5,
        capthick=1.2,
        alpha=0.85,
        zorder=2,
    )

    # Point moyen mis en évidence
    ax.scatter(
        mean_time,
        mean_f1,
        s=230,
        marker=marker,
        facecolor=color,
        edgecolor="black",
        linewidth=1.6,
        alpha=1.0,
        zorder=5,
    )

    ax.annotate(
        f"Moyenne\n{mean_f1:.2f} %",
        xy=(mean_time, mean_f1),
        xytext=(9, 10),
        textcoords="offset points",
        fontsize=8,
        fontweight="semibold",
        color=color,
        bbox={
            "boxstyle": "round,pad=0.25",
            "facecolor": "white",
            "edgecolor": color,
            "alpha": 0.85,
        },
        zorder=6,
    )

# ============================================================
# Mise en forme
# ============================================================

ax.set_xlabel(
    "Temps d'entraînement du SLM (minutes)",
    fontsize=12,
    fontweight="semibold",
    labelpad=10,
)

ax.set_ylabel(
    "Weighted-F1 final (%)",
    fontsize=12,
    fontweight="semibold",
    labelpad=10,
)

ax.set_title(
    "Compromis entre performance et temps d'entraînement",
    fontsize=16,
    fontweight="bold",
    pad=18,
)

ax.text(
    0.5,
    1.01,
    "Les grands marqueurs représentent la moyenne ; "
    "les barres indiquent ± un écart-type",
    transform=ax.transAxes,
    ha="center",
    va="bottom",
    fontsize=9.5,
    alpha=0.72,
)

# Format en pourcentage sans multiplier les valeurs
ax.yaxis.set_major_formatter(
    PercentFormatter(xmax=100, decimals=0)
)

# Marges automatiques pour éviter les points collés aux axes
x_min = eff_df["train_minutes"].min()
x_max = eff_df["train_minutes"].max()

y_min = eff_df["weighted_f1_pct"].min()
y_max = eff_df["weighted_f1_pct"].max()

x_margin = max(
    0.5,
    0.08 * max(x_max - x_min, 1.0),
)

y_margin = max(
    1.0,
    0.12 * max(y_max - y_min, 1.0),
)

ax.set_xlim(
    max(0.0, x_min - x_margin),
    x_max + x_margin,
)

ax.set_ylim(
    max(0.0, y_min - y_margin),
    min(101.0, y_max + y_margin),
)

# Grille légère
ax.grid(
    axis="both",
    linestyle="--",
    linewidth=0.7,
    alpha=0.28,
    zorder=0,
)

ax.set_axisbelow(True)

# Bordures simplifiées
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.spines["left"].set_alpha(0.45)
ax.spines["bottom"].set_alpha(0.45)

ax.tick_params(
    axis="both",
    labelsize=9.5,
)

# Légende sous la figure
ax.legend(
    title="Configuration",
    loc="upper center",
    bbox_to_anchor=(0.5, -0.15),
    ncol=min(3, len(grouped_efficiency)),
    frameon=False,
    fontsize=9,
    title_fontsize=10,
)

plt.tight_layout(rect=[0, 0.08, 1, 1])

save_figure(
    "comparative_efficiency_vs_weighted_f1"
)

plt.show()
plt.close(fig)

## 11 — Graphe : courbes d’apprentissage MiniLM

In [ ]:
for (protocol, backbone), subset_group in histories_df.groupby(
    ["protocol", "backbone"]
):
    display_name = backbone_display_name(backbone)

    plt.figure(figsize=(12, 5))
    for seed in sorted(subset_group["seed"].unique()):
        subset = subset_group[subset_group["seed"] == seed]
        plt.plot(
            subset["epoch"],
            subset["train_loss"],
            marker="o",
            label=f"seed={seed}",
        )
    plt.xlabel("Époque")
    plt.ylabel("Loss")
    plt.title(f"{display_name} — loss d'entraînement ({protocol})")
    plt.legend()
    save_figure(f"training_loss_{protocol}_{backbone}")
    plt.show()

    plt.figure(figsize=(12, 5))
    for seed in sorted(subset_group["seed"].unique()):
        subset = subset_group[subset_group["seed"] == seed]
        plt.plot(
            subset["epoch"],
            subset["val_bal_macro_f1"],
            marker="o",
            label=f"seed={seed}",
        )
    plt.xlabel("Époque")
    plt.ylabel("Macro-F1")
    plt.ylim(0.0, 1.02)
    plt.title(
        f"{display_name} — Macro-F1 validation équilibrée ({protocol})"
    )
    plt.legend()
    save_figure(f"training_macro_f1_{protocol}_{backbone}")
    plt.show()

In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator


# ============================================================
# Fonctions utilitaires
# ============================================================

def sanitize_filename(value):
    """Rend une chaîne compatible avec un nom de fichier."""
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value)).strip("_")


def prepare_history_group(subset_group):
    """Nettoie et trie les historiques d'entraînement."""
    required_columns = [
        "seed",
        "epoch",
        "train_loss",
        "val_bal_macro_f1",
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in subset_group.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Colonnes manquantes dans histories_df : {missing_columns}"
        )

    return (
        subset_group
        .dropna(subset=["seed", "epoch"])
        .sort_values(["seed", "epoch"])
        .copy()
    )


def plot_training_history(
    subset_group,
    metric,
    ylabel,
    title,
    output_name,
    ylim=None,
    highlight_best=False,
):
    """
    Affiche les courbes par seed, la moyenne inter-seeds
    et la bande moyenne ± écart-type.
    """

    subset_group = prepare_history_group(subset_group)

    seeds = sorted(subset_group["seed"].unique())
    color_map = plt.get_cmap("tab10")

    fig, ax = plt.subplots(figsize=(12, 6.3))

    # --------------------------------------------------------
    # Courbes individuelles par seed
    # --------------------------------------------------------

    for index, seed in enumerate(seeds):
        seed_df = (
            subset_group[subset_group["seed"] == seed]
            .dropna(subset=[metric])
            .sort_values("epoch")
        )

        if seed_df.empty:
            continue

        ax.plot(
            seed_df["epoch"],
            seed_df[metric],
            marker="o",
            markersize=5,
            markeredgecolor="white",
            markeredgewidth=0.7,
            linewidth=1.6,
            alpha=0.72,
            color=color_map(index % color_map.N),
            label=f"Seed {int(seed)}",
            zorder=2,
        )

    # --------------------------------------------------------
    # Moyenne et dispersion entre les seeds
    # --------------------------------------------------------

    epoch_summary = (
        subset_group
        .dropna(subset=[metric])
        .groupby("epoch", as_index=False)[metric]
        .agg(
            mean="mean",
            std="std",
            count="count",
        )
        .sort_values("epoch")
    )

    epoch_summary["std"] = epoch_summary["std"].fillna(0.0)

    if not epoch_summary.empty:
        epochs = epoch_summary["epoch"].to_numpy(dtype=float)
        means = epoch_summary["mean"].to_numpy(dtype=float)
        stds = epoch_summary["std"].to_numpy(dtype=float)

        lower_bound = means - stds
        upper_bound = means + stds

        if ylim is not None:
            lower_bound = np.maximum(lower_bound, ylim[0])
            upper_bound = np.minimum(upper_bound, ylim[1])

        # Zone ± écart-type
        if len(seeds) > 1:
            ax.fill_between(
                epochs,
                lower_bound,
                upper_bound,
                alpha=0.15,
                label="Moyenne ± écart-type",
                zorder=1,
            )

        # Courbe moyenne principale
        ax.plot(
            epochs,
            means,
            marker="D",
            markersize=6.5,
            markeredgecolor="white",
            markeredgewidth=0.9,
            linewidth=3.0,
            label="Moyenne inter-seeds",
            zorder=4,
        )

        # ----------------------------------------------------
        # Mise en évidence de la meilleure époque
        # ----------------------------------------------------

        if highlight_best:
            best_index = epoch_summary["mean"].idxmax()
            best_row = epoch_summary.loc[best_index]

            best_epoch = best_row["epoch"]
            best_value = best_row["mean"]

            ax.scatter(
                best_epoch,
                best_value,
                s=170,
                marker="*",
                edgecolor="black",
                linewidth=1.0,
                zorder=6,
            )

            ax.annotate(
                (
                    f"Meilleure moyenne\n"
                    f"Époque {int(best_epoch)} : {best_value:.3f}"
                ),
                xy=(best_epoch, best_value),
                xytext=(12, 18),
                textcoords="offset points",
                fontsize=9,
                fontweight="semibold",
                ha="left",
                va="bottom",
                bbox={
                    "boxstyle": "round,pad=0.35",
                    "facecolor": "white",
                    "edgecolor": "black",
                    "alpha": 0.88,
                },
                arrowprops={
                    "arrowstyle": "->",
                    "linewidth": 0.9,
                },
                zorder=7,
            )

    # --------------------------------------------------------
    # Mise en forme générale
    # --------------------------------------------------------

    ax.set_xlabel(
        "Époque",
        fontsize=12,
        fontweight="semibold",
        labelpad=9,
    )

    ax.set_ylabel(
        ylabel,
        fontsize=12,
        fontweight="semibold",
        labelpad=9,
    )

    ax.set_title(
        title,
        fontsize=15,
        fontweight="bold",
        pad=16,
    )

    ax.xaxis.set_major_locator(
        MaxNLocator(integer=True)
    )

    if ylim is not None:
        ax.set_ylim(*ylim)

    ax.grid(
        axis="both",
        linestyle="--",
        linewidth=0.7,
        alpha=0.28,
        zorder=0,
    )

    ax.set_axisbelow(True)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.spines["left"].set_alpha(0.45)
    ax.spines["bottom"].set_alpha(0.45)

    ax.tick_params(
        axis="both",
        labelsize=9.5,
    )

    ax.legend(
        title="Exécutions",
        loc="upper center",
        bbox_to_anchor=(0.5, -0.15),
        ncol=min(4, len(seeds) + 2),
        frameon=False,
        fontsize=9,
        title_fontsize=10,
    )

    plt.tight_layout(rect=[0, 0.08, 1, 1])

    save_figure(output_name)

    plt.show()
    plt.close(fig)


# ============================================================
# Génération des figures
# ============================================================

for (protocol, backbone), subset_group in histories_df.groupby(
    ["protocol", "backbone"],
    sort=True,
    observed=True,
):
    display_name = backbone_display_name(backbone)

    safe_protocol = sanitize_filename(protocol)
    safe_backbone = sanitize_filename(backbone)

    # --------------------------------------------------------
    # 1. Loss d'entraînement
    # --------------------------------------------------------

    plot_training_history(
        subset_group=subset_group,
        metric="train_loss",
        ylabel="Loss d'entraînement",
        title=(
            f"{display_name} — évolution de la loss d'entraînement\n"
            f"Protocole : {protocol}"
        ),
        output_name=(
            f"training_loss_{safe_protocol}_{safe_backbone}"
        ),
        ylim=None,
        highlight_best=False,
    )

    # --------------------------------------------------------
    # 2. Macro-F1 de validation
    # --------------------------------------------------------

    plot_training_history(
        subset_group=subset_group,
        metric="val_bal_macro_f1",
        ylabel="Macro-F1 de validation",
        title=(
            f"{display_name} — performance de validation équilibrée\n"
            f"Protocole : {protocol}"
        ),
        output_name=(
            f"training_macro_f1_{safe_protocol}_{safe_backbone}"
        ),
        ylim=(0.0, 1.02),
        highlight_best=True,
    )

## 12 — Graphes : audit des tokens et distribution des splits

In [ ]:
for (protocol, backbone), subset_group in token_audits_df.groupby(
    ["protocol", "backbone"]
):
    representative = subset_group.sort_values("seed").iloc[0]
    values = [
        representative["mean_tokens"],
        representative["median_tokens"],
        representative["p95_tokens"],
        representative["max_tokens"],
    ]

    plt.figure(figsize=(8, 4.5))
    plt.bar(["mean", "median", "p95", "max"], values)
    plt.axhline(
        float(representative["max_length"]),
        linestyle="--",
        label="max_length",
    )
    plt.ylabel("Nombre de tokens")
    plt.title(
        f"Audit tokens — {backbone_display_name(backbone)} | {protocol}"
    )
    plt.legend()
    save_figure(f"token_audit_{protocol}_{backbone}")
    plt.show()

for (protocol, backbone), subset in split_distributions_df.groupby(
    ["protocol", "backbone"]
):
    pivot = (
        subset.groupby(["split", "label"], as_index=False)["count"]
        .mean()
        .pivot(index="label", columns="split", values="count")
        .fillna(0.0)
    )
    pivot = pivot.loc[pivot.sum(axis=1).sort_values(ascending=False).index]

    plt.figure(figsize=(13, 6))
    x = np.arange(len(pivot.index))
    columns = pivot.columns.tolist()
    width = 0.8 / len(columns)

    for i, split_name in enumerate(columns):
        plt.bar(
            x + (i - (len(columns) - 1) / 2) * width,
            pivot[split_name].to_numpy(),
            width=width,
            label=split_name,
        )

    plt.xticks(x, pivot.index, rotation=45, ha="right")
    plt.ylabel("Nombre moyen de flux")
    plt.title(
        f"Distribution des splits — "
        f"{backbone_display_name(backbone)} | {protocol}"
    )
    plt.legend()
    save_figure(f"split_distribution_{protocol}_{backbone}")
    plt.show()

In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import (
    FuncFormatter,
    LogLocator,
    NullFormatter,
)


# ============================================================
# Fonctions utilitaires
# ============================================================

def sanitize_filename(value):
    """Nettoie une valeur pour produire un nom de fichier valide."""
    return re.sub(
        r"[^A-Za-z0-9_.-]+",
        "_",
        str(value),
    ).strip("_")


def compact_number(value, _=None):
    """Format compact : 1 200 → 1.2 k, 2 500 000 → 2.5 M."""
    abs_value = abs(value)

    if abs_value >= 1_000_000:
        return f"{value / 1_000_000:.1f} M"

    if abs_value >= 1_000:
        return f"{value / 1_000:.1f} k"

    return f"{value:.0f}"


# Palette cohérente entre toutes les figures
split_colors = {
    "train": "#4C78A8",
    "validation": "#F58518",
    "val": "#F58518",
    "calibration": "#54A24B",
    "cal": "#54A24B",
    "test": "#E45756",
}

split_display_names = {
    "train": "Entraînement",
    "validation": "Validation",
    "val": "Validation",
    "calibration": "Calibration",
    "cal": "Calibration",
    "test": "Test",
}


# ============================================================
# 1. Audit de la longueur des séquences tokenisées
# ============================================================

token_stat_columns = {
    "mean_tokens": "Moyenne",
    "median_tokens": "Médiane",
    "p95_tokens": "95e percentile",
    "max_tokens": "Maximum",
}

for (protocol, backbone), subset_group in token_audits_df.groupby(
    ["protocol", "backbone"],
    sort=True,
    observed=True,
):
    display_name = backbone_display_name(backbone)

    safe_protocol = sanitize_filename(protocol)
    safe_backbone = sanitize_filename(backbone)

    subset_group = subset_group.copy()

    # Moyenne et écart-type entre les seeds
    token_summary = (
        subset_group[list(token_stat_columns.keys())]
        .agg(["mean", "std"])
        .T
        .reset_index()
        .rename(columns={"index": "statistic"})
    )

    token_summary["std"] = token_summary["std"].fillna(0.0)

    token_summary["label"] = token_summary["statistic"].map(
        token_stat_columns
    )

    # Conserver un ordre logique
    statistic_order = list(token_stat_columns.keys())

    token_summary["statistic"] = (
        token_summary["statistic"]
        .astype("category")
        .cat.set_categories(
            statistic_order,
            ordered=True,
        )
    )

    token_summary = token_summary.sort_values("statistic")

    # max_length peut théoriquement être identique pour tous les seeds
    max_length = float(
        subset_group["max_length"].median()
    )

    means = token_summary["mean"].to_numpy(dtype=float)
    stds = token_summary["std"].to_numpy(dtype=float)
    labels = token_summary["label"].tolist()

    y_positions = np.arange(len(labels))

    fig, ax = plt.subplots(figsize=(10.5, 5.8))

    colors = [
        "#8ECAE6",
        "#5FA8D3",
        "#3572A5",
        "#173F5F",
    ]

    bars = ax.barh(
        y_positions,
        means,
        xerr=stds,
        height=0.64,
        color=colors,
        edgecolor="white",
        linewidth=1.0,
        error_kw={
            "elinewidth": 1.2,
            "capsize": 4,
            "capthick": 1.1,
        },
        zorder=3,
    )

    # Ligne représentant la limite de tokenisation
    ax.axvline(
        max_length,
        color="#C44E52",
        linestyle="--",
        linewidth=1.8,
        label=f"Longueur maximale = {max_length:.0f}",
        zorder=2,
    )

    # Annotations : nombre de tokens et pourcentage de max_length
    annotation_margin = max(
        max_length * 0.02,
        max(means) * 0.02,
        1.0,
    )

    for bar, mean_value, std_value in zip(
        bars,
        means,
        stds,
    ):
        percentage = (
            100.0 * mean_value / max_length
            if max_length > 0
            else np.nan
        )

        ax.text(
            mean_value + std_value + annotation_margin,
            bar.get_y() + bar.get_height() / 2,
            f"{mean_value:.1f} tokens  ({percentage:.1f} %)",
            va="center",
            ha="left",
            fontsize=9,
            fontweight="medium",
        )

    ax.set_yticks(y_positions)
    ax.set_yticklabels(
        labels,
        fontsize=10,
        fontweight="medium",
    )

    # La moyenne apparaît en haut et le maximum en bas
    ax.invert_yaxis()

    ax.set_xlabel(
        "Nombre de tokens",
        fontsize=11.5,
        fontweight="semibold",
        labelpad=10,
    )

    ax.set_ylabel("")

    ax.set_title(
        f"Audit des longueurs de séquence — {display_name}",
        fontsize=15,
        fontweight="bold",
        pad=17,
    )

    ax.text(
        0.5,
        1.01,
        f"Protocole : {protocol} · moyenne ± écart-type entre les seeds",
        transform=ax.transAxes,
        ha="center",
        va="bottom",
        fontsize=9.5,
        alpha=0.72,
    )

    maximum_value = max(
        max_length,
        np.max(means + stds),
    )

    ax.set_xlim(
        0,
        maximum_value * 1.30,
    )

    ax.xaxis.set_major_formatter(
        FuncFormatter(compact_number)
    )

    ax.grid(
        axis="x",
        linestyle="--",
        linewidth=0.7,
        alpha=0.28,
        zorder=0,
    )

    ax.set_axisbelow(True)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)

    ax.spines["bottom"].set_alpha(0.45)

    ax.tick_params(
        axis="y",
        length=0,
    )

    ax.legend(
        loc="lower right",
        frameon=False,
        fontsize=9.5,
    )

    plt.tight_layout()

    save_figure(
        f"token_audit_{safe_protocol}_{safe_backbone}"
    )

    plt.show()
    plt.close(fig)


# ============================================================
# 2. Distribution des classes dans les splits
# ============================================================

for (protocol, backbone), subset in split_distributions_df.groupby(
    ["protocol", "backbone"],
    sort=True,
    observed=True,
):
    display_name = backbone_display_name(backbone)

    safe_protocol = sanitize_filename(protocol)
    safe_backbone = sanitize_filename(backbone)

    # Moyenne entre les seeds pour chaque split et chaque classe
    pivot = (
        subset
        .groupby(
            ["split", "label"],
            as_index=False,
            observed=True,
        )["count"]
        .mean()
        .pivot(
            index="label",
            columns="split",
            values="count",
        )
        .fillna(0.0)
    )

    # Trier les classes par effectif total décroissant
    class_order = (
        pivot.sum(axis=1)
        .sort_values(ascending=False)
        .index
    )

    pivot = pivot.loc[class_order]

    # Ordre préférentiel des splits
    preferred_split_order = [
        "train",
        "validation",
        "val",
        "calibration",
        "cal",
        "test",
    ]

    split_columns = [
        split_name
        for split_name in preferred_split_order
        if split_name in pivot.columns
    ]

    # Ajouter d'éventuels splits non prévus
    split_columns += [
        split_name
        for split_name in pivot.columns
        if split_name not in split_columns
    ]

    pivot = pivot[split_columns]

    n_classes = len(pivot.index)
    n_splits = max(1, len(split_columns))

    y = np.arange(n_classes)

    total_group_height = 0.82
    bar_height = total_group_height / n_splits

    figure_height = max(
        6.5,
        0.55 * n_classes + 2.5,
    )

    fig, ax = plt.subplots(
        figsize=(13, figure_height)
    )

    cmap = plt.get_cmap("tab10")

    for index, split_name in enumerate(split_columns):
        positions = (
            y
            + (
                index
                - (n_splits - 1) / 2
            )
            * bar_height
        )

        color = split_colors.get(
            str(split_name).lower(),
            cmap(index % cmap.N),
        )

        display_split = split_display_names.get(
            str(split_name).lower(),
            str(split_name).replace("_", " ").title(),
        )

        values = pivot[split_name].to_numpy(dtype=float)

        bars = ax.barh(
            positions,
            values,
            height=bar_height * 0.88,
            color=color,
            edgecolor="white",
            linewidth=0.7,
            alpha=0.90,
            label=display_split,
            zorder=3,
        )

        # Annotation seulement lorsque le graphique n'est pas trop chargé
        if n_classes <= 12:
            maximum_count = pivot.to_numpy().max()

            for bar, value in zip(bars, values):
                if value <= 0:
                    continue

                ax.text(
                    value + maximum_count * 0.008,
                    bar.get_y() + bar.get_height() / 2,
                    compact_number(value),
                    va="center",
                    ha="left",
                    fontsize=7.5,
                    alpha=0.85,
                )

    ax.set_yticks(y)

    ax.set_yticklabels(
        pivot.index,
        fontsize=9.5,
    )

    # Classe majoritaire en haut
    ax.invert_yaxis()

    ax.set_xlabel(
        "Nombre moyen de flux",
        fontsize=11.5,
        fontweight="semibold",
        labelpad=10,
    )

    ax.set_ylabel(
        "Classe",
        fontsize=11.5,
        fontweight="semibold",
        labelpad=10,
    )

    ax.set_title(
        f"Distribution des classes dans les partitions — {display_name}",
        fontsize=15,
        fontweight="bold",
        pad=17,
    )

    ax.text(
        0.5,
        1.01,
        f"Protocole : {protocol} · effectifs moyens entre les seeds",
        transform=ax.transAxes,
        ha="center",
        va="bottom",
        fontsize=9.5,
        alpha=0.72,
    )

    # Détecter automatiquement un fort déséquilibre
    positive_values = pivot.to_numpy()
    positive_values = positive_values[
        positive_values > 0
    ]

    use_log_scale = False

    if positive_values.size > 0:
        imbalance_ratio = (
            positive_values.max()
            / positive_values.min()
        )

        use_log_scale = imbalance_ratio >= 100

    if use_log_scale:
        ax.set_xscale("log")

        ax.xaxis.set_major_locator(
            LogLocator(base=10)
        )

        ax.xaxis.set_minor_formatter(
            NullFormatter()
        )

        ax.text(
            0.99,
            0.02,
            "Échelle logarithmique",
            transform=ax.transAxes,
            ha="right",
            va="bottom",
            fontsize=8.5,
            fontstyle="italic",
            alpha=0.65,
        )

    else:
        ax.xaxis.set_major_formatter(
            FuncFormatter(compact_number)
        )

        maximum_count = pivot.to_numpy().max()

        ax.set_xlim(
            0,
            maximum_count * 1.16,
        )

    ax.grid(
        axis="x",
        which="major",
        linestyle="--",
        linewidth=0.7,
        alpha=0.27,
        zorder=0,
    )

    ax.set_axisbelow(True)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.spines["left"].set_alpha(0.40)
    ax.spines["bottom"].set_alpha(0.40)

    ax.tick_params(
        axis="y",
        length=0,
    )

    ax.legend(
        title="Partition",
        loc="upper center",
        bbox_to_anchor=(0.5, -0.10),
        ncol=min(4, n_splits),
        frameon=False,
        fontsize=9,
        title_fontsize=10,
    )

    plt.tight_layout(
        rect=[0, 0.05, 1, 1]
    )

    save_figure(
        f"split_distribution_{safe_protocol}_{safe_backbone}"
    )

    plt.show()
    plt.close(fig)

## 13 — Graphes : matrices de confusion des meilleurs runs

In [ ]:
for backbone in CFG.backbones:
    for protocol in CFG.protocols:
        subset = summaries_df[
            (summaries_df["protocol"] == protocol)
            & (summaries_df["backbone"] == backbone)
        ]
        if subset.empty:
            continue
        best_row = subset.loc[subset["final_weighted_f1"].idxmax()]
        key = (protocol, backbone, int(best_row["seed"]))
        cm_info = confusion_registry[key]

        plt.figure(figsize=(10, 8))
        sns.heatmap(
            cm_info["cm"],
            annot=False,
            fmt="d",
            xticklabels=cm_info["class_names"],
            yticklabels=cm_info["class_names"],
            cbar=True,
        )
        plt.xlabel("Prédit")
        plt.ylabel("Vrai")
        plt.title(
            f"Matrice de confusion — {backbone_display_name(backbone)} | "
            f"{protocol} | seed={int(best_row['seed'])}"
        )
        save_figure(f"confusion_best_{protocol}_{backbone}")
        plt.show()

In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# Fonctions utilitaires
# ============================================================

def sanitize_filename(value):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value)).strip("_")


def plot_pretty_confusion_matrix(
    cm,
    class_names,
    title,
    output_name,
    cmap="Blues",
    normalize=True,
):
    """
    Affiche une matrice de confusion plus élégante.
    - normalize=True : affichage principal en pourcentage par ligne
    - annotations = effectif + pourcentage
    """
    cm = np.asarray(cm, dtype=float)
    n_classes = len(class_names)

    row_sums = cm.sum(axis=1, keepdims=True)
    cm_norm = np.divide(
        cm,
        row_sums,
        out=np.zeros_like(cm, dtype=float),
        where=row_sums != 0,
    )

    # Taille adaptative
    fig_width = max(9, min(16, 0.75 * n_classes + 4))
    fig_height = max(7, min(14, 0.70 * n_classes + 3))

    fig, ax = plt.subplots(figsize=(fig_width, fig_height))

    matrix_to_show = cm_norm if normalize else cm
    im = ax.imshow(matrix_to_show, interpolation="nearest", cmap=cmap, aspect="auto")

    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.ax.tick_params(labelsize=9)

    if normalize:
        cbar.set_label("Proportion par classe vraie", fontsize=10)
    else:
        cbar.set_label("Nombre d'exemples", fontsize=10)

    ax.set_xticks(np.arange(n_classes))
    ax.set_yticks(np.arange(n_classes))

    ax.set_xticklabels(
        class_names,
        rotation=45,
        ha="right",
        rotation_mode="anchor",
        fontsize=9,
    )
    ax.set_yticklabels(
        class_names,
        fontsize=9,
    )

    ax.set_xlabel(
        "Classe prédite",
        fontsize=12,
        fontweight="semibold",
        labelpad=12,
    )
    ax.set_ylabel(
        "Classe vraie",
        fontsize=12,
        fontweight="semibold",
        labelpad=12,
    )

    ax.set_title(
        title,
        fontsize=15,
        fontweight="bold",
        pad=18,
    )

    # Sous-grille légère
    ax.set_xticks(np.arange(-0.5, n_classes, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, n_classes, 1), minor=True)
    ax.grid(which="minor", color="white", linestyle="-", linewidth=1.0)
    ax.tick_params(which="minor", bottom=False, left=False)

    # Annotations
    threshold = matrix_to_show.max() / 2.0 if matrix_to_show.size else 0.0

    for i in range(n_classes):
        for j in range(n_classes):
            count_value = int(cm[i, j])
            pct_value = 100.0 * cm_norm[i, j]

            # Ne rien afficher pour les zéros si vous voulez un rendu plus léger
            if count_value == 0:
                annotation = ""
            else:
                annotation = f"{count_value}\n({pct_value:.1f}%)"

            text_color = "white" if matrix_to_show[i, j] > threshold else "black"

            ax.text(
                j,
                i,
                annotation,
                ha="center",
                va="center",
                color=text_color,
                fontsize=8.5,
                fontweight="medium",
            )

    # Bordures propres
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_alpha(0.4)
    ax.spines["bottom"].set_alpha(0.4)

    plt.tight_layout()
    save_figure(output_name)
    plt.show()
    plt.close(fig)


# ============================================================
# Génération des matrices de confusion
# ============================================================

for backbone in CFG.backbones:
    for protocol in CFG.protocols:
        subset = summaries_df[
            (summaries_df["protocol"] == protocol)
            & (summaries_df["backbone"] == backbone)
        ]

        if subset.empty:
            continue

        best_row = subset.loc[subset["final_weighted_f1"].idxmax()]
        best_seed = int(best_row["seed"])

        key = (protocol, backbone, best_seed)
        cm_info = confusion_registry[key]

        display_name = backbone_display_name(backbone)

        weighted_f1 = best_row.get("final_weighted_f1", np.nan)
        macro_f1 = best_row.get("final_macro_f1", np.nan)
        bal_acc = best_row.get("final_balanced_accuracy", np.nan)

        subtitle_parts = []
        if not np.isnan(weighted_f1):
            subtitle_parts.append(f"Weighted-F1={weighted_f1:.3f}")
        if not np.isnan(macro_f1):
            subtitle_parts.append(f"Macro-F1={macro_f1:.3f}")
        if not np.isnan(bal_acc):
            subtitle_parts.append(f"Balanced Acc.={bal_acc:.3f}")

        subtitle = " | ".join(subtitle_parts)

        title = (
            f"Matrice de confusion — {display_name} | {protocol} | seed={best_seed}"
        )
        if subtitle:
            title += f"\n{subtitle}"

        plot_pretty_confusion_matrix(
            cm=cm_info["cm"],
            class_names=cm_info["class_names"],
            title=title,
            output_name=f"confusion_best_{sanitize_filename(protocol)}_{sanitize_filename(backbone)}",
            cmap="Blues",
            normalize=True,
        )

In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter


# ============================================================
# Fonctions utilitaires
# ============================================================

def sanitize_filename(value):
    """Nettoie une chaîne pour l'utiliser dans un nom de fichier."""
    return re.sub(
        r"[^A-Za-z0-9_.-]+",
        "_",
        str(value),
    ).strip("_")


def safe_metric(row, possible_names):
    """
    Recherche une métrique dans plusieurs noms de colonnes possibles.
    Retourne NaN si aucune colonne n'est disponible.
    """
    for column in possible_names:
        if column in row.index:
            value = row[column]

            if value is not None and np.isfinite(value):
                return float(value)

    return np.nan


# ============================================================
# Fonction de visualisation
# ============================================================

def plot_pretty_confusion_matrix(
    cm,
    class_names,
    title,
    subtitle,
    output_name,
    normalize=True,
    show_zero_percentage=False,
    cmap="Blues",
):
    """
    Produit une matrice de confusion prête pour un article.

    Paramètres
    ----------
    cm :
        Matrice de confusion brute.

    class_names :
        Noms des classes dans l'ordre de la matrice.

    normalize :
        Si True, les couleurs représentent les proportions par ligne.

    show_zero_percentage :
        Si False, une cellule nulle affiche seulement "0".
        Si True, elle affiche "0\\n(0.0 %)".

    Les cellules non nulles affichent :
        effectif
        pourcentage par classe vraie
    """

    # --------------------------------------------------------
    # Vérification des données
    # --------------------------------------------------------

    cm = np.asarray(cm)

    if cm.ndim != 2:
        raise ValueError(
            "La matrice de confusion doit être bidimensionnelle."
        )

    if cm.shape[0] != cm.shape[1]:
        raise ValueError(
            f"La matrice doit être carrée, mais sa forme est {cm.shape}."
        )

    class_names = [str(name) for name in class_names]

    if len(class_names) != cm.shape[0]:
        raise ValueError(
            "Le nombre de noms de classes ne correspond pas "
            "aux dimensions de la matrice de confusion."
        )

    if np.any(cm < 0):
        raise ValueError(
            "La matrice de confusion contient des valeurs négatives."
        )

    cm_counts = cm.astype(np.int64)
    cm_float = cm_counts.astype(float)

    n_classes = len(class_names)

    # --------------------------------------------------------
    # Normalisation par classe vraie
    # --------------------------------------------------------

    row_sums = cm_float.sum(axis=1, keepdims=True)

    cm_normalized = np.divide(
        cm_float,
        row_sums,
        out=np.zeros_like(cm_float),
        where=row_sums > 0,
    )

    displayed_matrix = (
        cm_normalized
        if normalize
        else cm_float
    )

    # --------------------------------------------------------
    # Taille dynamique de la figure
    # --------------------------------------------------------

    figure_width = max(
        10,
        min(18, 0.90 * n_classes + 5),
    )

    figure_height = max(
        8,
        min(16, 0.78 * n_classes + 4),
    )

    fig, ax = plt.subplots(
        figsize=(figure_width, figure_height)
    )

    # --------------------------------------------------------
    # Affichage de la matrice
    # --------------------------------------------------------

    image = ax.imshow(
        displayed_matrix,
        interpolation="nearest",
        cmap=cmap,
        aspect="equal",
        vmin=0,
        vmax=1 if normalize else None,
    )

    # --------------------------------------------------------
    # Barre de couleur
    # --------------------------------------------------------

    colorbar = fig.colorbar(
        image,
        ax=ax,
        fraction=0.046,
        pad=0.04,
    )

    if normalize:
        colorbar.ax.yaxis.set_major_formatter(
            PercentFormatter(
                xmax=1.0,
                decimals=0,
            )
        )

        colorbar.set_label(
            "Proportion parmi les exemples de la classe vraie",
            fontsize=10,
            fontweight="semibold",
            labelpad=12,
        )

    else:
        colorbar.set_label(
            "Nombre d'exemples",
            fontsize=10,
            fontweight="semibold",
            labelpad=12,
        )

    colorbar.ax.tick_params(labelsize=9)

    # --------------------------------------------------------
    # Axes et noms de classes
    # --------------------------------------------------------

    positions = np.arange(n_classes)

    ax.set_xticks(positions)
    ax.set_yticks(positions)

    ax.set_xticklabels(
        class_names,
        rotation=45,
        ha="right",
        rotation_mode="anchor",
        fontsize=9,
    )

    ax.set_yticklabels(
        class_names,
        fontsize=9,
    )

    ax.set_xlabel(
        "Classe prédite",
        fontsize=12,
        fontweight="semibold",
        labelpad=14,
    )

    ax.set_ylabel(
        "Classe vraie",
        fontsize=12,
        fontweight="semibold",
        labelpad=14,
    )

    # --------------------------------------------------------
    # Titre et sous-titre
    # --------------------------------------------------------

    ax.set_title(
        title,
        fontsize=15,
        fontweight="bold",
        pad=28,
    )

    if subtitle:
        ax.text(
            0.5,
            1.015,
            subtitle,
            transform=ax.transAxes,
            ha="center",
            va="bottom",
            fontsize=9.5,
            alpha=0.76,
        )

    # --------------------------------------------------------
    # Séparation visuelle entre les cellules
    # --------------------------------------------------------

    ax.set_xticks(
        np.arange(-0.5, n_classes, 1),
        minor=True,
    )

    ax.set_yticks(
        np.arange(-0.5, n_classes, 1),
        minor=True,
    )

    ax.grid(
        which="minor",
        color="white",
        linestyle="-",
        linewidth=1.2,
    )

    ax.tick_params(
        which="minor",
        bottom=False,
        left=False,
    )

    ax.tick_params(
        axis="both",
        which="major",
        length=0,
    )

    # --------------------------------------------------------
    # Annotations dans toutes les cellules
    # --------------------------------------------------------

    maximum_value = (
        displayed_matrix.max()
        if displayed_matrix.size > 0
        else 0.0
    )

    # Seuil pour choisir automatiquement noir ou blanc
    text_threshold = (
        0.50 * maximum_value
        if maximum_value > 0
        else 0.0
    )

    # Taille du texte adaptée au nombre de classes
    if n_classes <= 8:
        annotation_fontsize = 8.5
    elif n_classes <= 12:
        annotation_fontsize = 7.5
    else:
        annotation_fontsize = 6.5

    for true_index in range(n_classes):
        for predicted_index in range(n_classes):

            count_value = int(
                cm_counts[true_index, predicted_index]
            )

            percentage_value = (
                100.0
                * cm_normalized[
                    true_index,
                    predicted_index,
                ]
            )

            # Les zéros restent visibles
            if count_value == 0:
                if show_zero_percentage:
                    annotation = "0\n(0.0 %)"
                else:
                    annotation = "0"
            else:
                annotation = (
                    f"{count_value:,}\n"
                    f"({percentage_value:.1f} %)"
                )

            cell_value = displayed_matrix[
                true_index,
                predicted_index
            ]

            text_color = (
                "white"
                if cell_value > text_threshold
                else "black"
            )

            # Cellules diagonales en gras
            font_weight = (
                "bold"
                if true_index == predicted_index
                else "normal"
            )

            ax.text(
                predicted_index,
                true_index,
                annotation,
                ha="center",
                va="center",
                color=text_color,
                fontsize=annotation_fontsize,
                fontweight=font_weight,
                linespacing=1.15,
            )

    # --------------------------------------------------------
    # Encadrement léger de la diagonale
    # --------------------------------------------------------

    for index in range(n_classes):
        diagonal_rectangle = plt.Rectangle(
            (index - 0.5, index - 0.5),
            1,
            1,
            fill=False,
            edgecolor="black",
            linewidth=1.15,
            alpha=0.55,
        )

        ax.add_patch(diagonal_rectangle)

    # --------------------------------------------------------
    # Style général
    # --------------------------------------------------------

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_alpha(0.35)
    ax.spines["bottom"].set_alpha(0.35)

    ax.set_xlim(-0.5, n_classes - 0.5)
    ax.set_ylim(n_classes - 0.5, -0.5)

    plt.tight_layout()

    save_figure(output_name)

    plt.show()
    plt.close(fig)


# ============================================================
# Génération des matrices pour chaque configuration
# ============================================================

for backbone in CFG.backbones:
    for protocol in CFG.protocols:

        subset = summaries_df[
            (summaries_df["protocol"] == protocol)
            & (summaries_df["backbone"] == backbone)
        ].copy()

        if subset.empty:
            continue

        # ----------------------------------------------------
        # Sélection du meilleur seed selon le Weighted-F1
        # ----------------------------------------------------

        valid_subset = subset.dropna(
            subset=["final_weighted_f1"]
        )

        if valid_subset.empty:
            print(
                f"Aucun Weighted-F1 valide pour "
                f"{protocol} / {backbone}."
            )
            continue

        best_index = valid_subset[
            "final_weighted_f1"
        ].idxmax()

        best_row = valid_subset.loc[best_index]

        best_seed = int(best_row["seed"])

        registry_key = (
            protocol,
            backbone,
            best_seed,
        )

        if registry_key not in confusion_registry:
            print(
                "Matrice introuvable pour : "
                f"{registry_key}"
            )
            continue

        cm_info = confusion_registry[registry_key]

        if "cm" not in cm_info:
            print(
                f"Clé 'cm' absente pour {registry_key}."
            )
            continue

        if "class_names" not in cm_info:
            print(
                f"Clé 'class_names' absente pour {registry_key}."
            )
            continue

        # ----------------------------------------------------
        # Métriques affichées dans le sous-titre
        # ----------------------------------------------------

        weighted_f1 = safe_metric(
            best_row,
            [
                "final_weighted_f1",
                "weighted_f1",
            ],
        )

        macro_f1 = safe_metric(
            best_row,
            [
                "final_macro_f1",
                "macro_f1",
            ],
        )

        macro_f1_all = safe_metric(
            best_row,
            [
                "final_macro_f1_all",
                "macro_f1_all",
            ],
        )

        balanced_accuracy = safe_metric(
            best_row,
            [
                "final_balanced_accuracy",
                "balanced_accuracy",
            ],
        )

        mcc = safe_metric(
            best_row,
            [
                "final_mcc",
                "mcc",
            ],
        )

        metric_parts = []

        if np.isfinite(weighted_f1):
            metric_parts.append(
                f"Weighted-F1 = {weighted_f1:.3f}"
            )

        if np.isfinite(macro_f1):
            metric_parts.append(
                f"Macro-F1 = {macro_f1:.3f}"
            )

        if np.isfinite(macro_f1_all):
            metric_parts.append(
                f"Macro-F1 global = {macro_f1_all:.3f}"
            )

        if np.isfinite(balanced_accuracy):
            metric_parts.append(
                f"Balanced Accuracy = "
                f"{balanced_accuracy:.3f}"
            )

        if np.isfinite(mcc):
            metric_parts.append(
                f"MCC = {mcc:.3f}"
            )

        display_name = backbone_display_name(
            backbone
        )

        title = (
            f"Matrice de confusion normalisée — "
            f"{display_name}"
        )

        subtitle_prefix = (
            f"Protocole : {protocol} · "
            f"meilleur seed : {best_seed}"
        )

        if metric_parts:
            subtitle = (
                subtitle_prefix
                + "\n"
                + " · ".join(metric_parts)
            )
        else:
            subtitle = subtitle_prefix

        output_name = (
            "confusion_best_"
            f"{sanitize_filename(protocol)}_"
            f"{sanitize_filename(backbone)}"
        )

        # ----------------------------------------------------
        # Création du graphique
        # ----------------------------------------------------

        plot_pretty_confusion_matrix(
            cm=cm_info["cm"],
            class_names=cm_info["class_names"],
            title=title,
            subtitle=subtitle,
            output_name=output_name,
            normalize=True,

            # False : les cellules nulles affichent seulement "0"
            # True  : elles affichent "0" et "(0.0 %)"
            show_zero_percentage=False,

            cmap="Blues",
        )

## 14 — Graphes : importance des caractéristiques XGBoost

In [ ]:
importance_mean = (
    feature_importances_df
    .groupby(["protocol", "backbone", "feature"], as_index=False)["importance"]
    .mean()
)

for (protocol, backbone), subset_group in importance_mean.groupby(
    ["protocol", "backbone"]
):
    subset = subset_group.sort_values("importance", ascending=False).head(20)
    plt.figure(figsize=(10, 7))
    plt.barh(subset["feature"][::-1], subset["importance"][::-1])
    plt.xlabel("Importance moyenne")
    plt.title(
        f"Top-20 variables — XGBoost associé à "
        f"{backbone_display_name(backbone)} | {protocol}"
    )
    save_figure(f"xgb_importance_{protocol}_{backbone}")
    plt.show()

In [ ]:
import re
import textwrap
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter


# ============================================================
# Fonctions utilitaires
# ============================================================

def sanitize_filename(value):
    """Produit un nom de fichier sûr."""
    return re.sub(
        r"[^A-Za-z0-9_.-]+",
        "_",
        str(value),
    ).strip("_")


def prettify_feature_name(feature_name, width=28):
    """
    Rend les noms de variables plus lisibles et les coupe
    sur plusieurs lignes lorsqu'ils sont trop longs.
    """
    readable_name = (
        str(feature_name)
        .replace("_", " ")
        .replace("-", " ")
        .strip()
    )

    return "\n".join(
        textwrap.wrap(
            readable_name,
            width=width,
            break_long_words=False,
        )
    )


# ============================================================
# Agrégation des importances entre les seeds
# ============================================================

importance_summary = (
    feature_importances_df
    .dropna(
        subset=[
            "protocol",
            "backbone",
            "feature",
            "importance",
        ]
    )
    .groupby(
        ["protocol", "backbone", "feature"],
        as_index=False,
        observed=True,
    )["importance"]
    .agg(
        importance_mean="mean",
        importance_std="std",
        n_runs="count",
    )
)

# Un seul seed produit un écart-type indéfini
importance_summary["importance_std"] = (
    importance_summary["importance_std"]
    .fillna(0.0)
)


# ============================================================
# Fonction de visualisation
# ============================================================

def plot_xgboost_feature_importance(
    data,
    protocol,
    backbone,
    top_k=20,
):
    display_name = backbone_display_name(backbone)

    subset = (
        data
        .sort_values(
            "importance_mean",
            ascending=False,
        )
        .head(top_k)
        .copy()
    )

    if subset.empty:
        return

    # Normalisation relative uniquement pour les annotations
    total_top_importance = subset["importance_mean"].sum()

    subset["relative_percentage"] = np.where(
        total_top_importance > 0,
        100.0
        * subset["importance_mean"]
        / total_top_importance,
        0.0,
    )

    # Rang global dans le Top-K
    subset["rank"] = np.arange(
        1,
        len(subset) + 1,
    )

    # Ordre inversé pour placer la variable la plus importante en haut
    plot_data = subset.iloc[::-1].copy()

    feature_labels = [
        f"{rank:02d}. {prettify_feature_name(feature)}"
        for rank, feature in zip(
            plot_data["rank"],
            plot_data["feature"],
        )
    ]

    means = plot_data[
        "importance_mean"
    ].to_numpy(dtype=float)

    stds = plot_data[
        "importance_std"
    ].to_numpy(dtype=float)

    percentages = plot_data[
        "relative_percentage"
    ].to_numpy(dtype=float)

    y_positions = np.arange(len(plot_data))

    # Taille dynamique selon le nombre de variables
    figure_height = max(
        7.5,
        0.48 * len(plot_data) + 2.5,
    )

    fig, ax = plt.subplots(
        figsize=(12.5, figure_height)
    )

    # Dégradé selon l'importance
    normalized_values = (
        means / means.max()
        if means.max() > 0
        else np.zeros_like(means)
    )

    colors = plt.cm.Blues(
        0.35 + 0.60 * normalized_values
    )

    bars = ax.barh(
        y_positions,
        means,
        xerr=stds,
        height=0.68,
        color=colors,
        edgecolor="white",
        linewidth=0.9,
        error_kw={
            "elinewidth": 1.1,
            "capsize": 4,
            "capthick": 1.0,
            "alpha": 0.75,
        },
        zorder=3,
    )

    # ========================================================
    # Annotations
    # ========================================================

    maximum_extent = np.max(means + stds)

    annotation_margin = max(
        maximum_extent * 0.015,
        0.0001,
    )

    for bar, mean_value, std_value, percentage in zip(
        bars,
        means,
        stds,
        percentages,
    ):
        ax.text(
            mean_value
            + std_value
            + annotation_margin,
            bar.get_y() + bar.get_height() / 2,
            f"{mean_value:.4f}  ({percentage:.1f} %)",
            va="center",
            ha="left",
            fontsize=8.5,
            fontweight="medium",
        )

    # ========================================================
    # Axes
    # ========================================================

    ax.set_yticks(y_positions)
    ax.set_yticklabels(
        feature_labels,
        fontsize=9.2,
        linespacing=1.15,
    )

    ax.set_xlabel(
        "Importance moyenne XGBoost",
        fontsize=11.5,
        fontweight="semibold",
        labelpad=11,
    )

    ax.set_ylabel(
        "Variables",
        fontsize=11.5,
        fontweight="semibold",
        labelpad=12,
    )

    ax.set_title(
        f"Top-{len(plot_data)} des variables les plus importantes",
        fontsize=15.5,
        fontweight="bold",
        pad=22,
    )

    ax.text(
        0.5,
        1.01,
        (
            f"XGBoost associé à {display_name} · "
            f"protocole : {protocol} · "
            f"moyenne ± écart-type entre les exécutions"
        ),
        transform=ax.transAxes,
        ha="center",
        va="bottom",
        fontsize=9.5,
        alpha=0.72,
    )

    # Format lisible de l'axe x
    ax.xaxis.set_major_formatter(
        FuncFormatter(
            lambda value, _: (
                f"{value:.3f}"
                if value < 1
                else f"{value:.1f}"
            )
        )
    )

    # Espace réservé aux annotations
    ax.set_xlim(
        0,
        maximum_extent * 1.32
        if maximum_extent > 0
        else 1,
    )

    # ========================================================
    # Mise en forme
    # ========================================================

    ax.grid(
        axis="x",
        linestyle="--",
        linewidth=0.7,
        alpha=0.28,
        zorder=0,
    )

    ax.set_axisbelow(True)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)
    ax.spines["bottom"].set_alpha(0.40)

    ax.tick_params(
        axis="y",
        length=0,
        pad=7,
    )

    ax.tick_params(
        axis="x",
        labelsize=9,
    )

    # Petite note explicative
    ax.text(
        0.99,
        0.01,
        "Valeur entre parenthèses : contribution relative au Top affiché",
        transform=ax.transAxes,
        ha="right",
        va="bottom",
        fontsize=8,
        fontstyle="italic",
        alpha=0.60,
    )

    plt.tight_layout()

    output_name = (
        f"xgb_importance_"
        f"{sanitize_filename(protocol)}_"
        f"{sanitize_filename(backbone)}"
    )

    save_figure(output_name)

    plt.show()
    plt.close(fig)


# ============================================================
# Génération des graphiques
# ============================================================

for (
    protocol,
    backbone,
), subset_group in importance_summary.groupby(
    ["protocol", "backbone"],
    sort=True,
    observed=True,
):
    plot_xgboost_feature_importance(
        data=subset_group,
        protocol=protocol,
        backbone=backbone,
        top_k=20,
    )

In [ ]:
import re
import textwrap
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter


# ============================================================
# Fonctions utilitaires
# ============================================================

def sanitize_filename(value):
    """Nettoie une chaîne pour produire un nom de fichier valide."""
    return re.sub(
        r"[^A-Za-z0-9_.-]+",
        "_",
        str(value),
    ).strip("_")


def prettify_feature_name(feature_name, width=32):
    """Transforme les noms techniques en libellés plus lisibles."""
    readable_name = (
        str(feature_name)
        .replace("_", " ")
        .replace("-", " ")
        .strip()
    )

    readable_name = " ".join(
        word.upper() if word.lower() in {
            "iat", "fwd", "bwd", "tcp", "udp", "ip", "ttl"
        }
        else word.capitalize()
        for word in readable_name.split()
    )

    return "\n".join(
        textwrap.wrap(
            readable_name,
            width=width,
            break_long_words=False,
        )
    )


# ============================================================
# Agrégation des importances entre les exécutions
# ============================================================

importance_summary = (
    feature_importances_df
    .dropna(
        subset=[
            "protocol",
            "backbone",
            "feature",
            "importance",
        ]
    )
    .groupby(
        ["protocol", "backbone", "feature"],
        as_index=False,
        observed=True,
    )
    .agg(
        importance_mean=("importance", "mean"),
        importance_std=("importance", "std"),
        n_runs=("importance", "count"),
    )
)

importance_summary["importance_std"] = (
    importance_summary["importance_std"]
    .fillna(0.0)
)


# ============================================================
# Graphique principal
# ============================================================

def plot_feature_importance_lollipop(
    data,
    protocol,
    backbone,
    top_k=20,
):
    display_name = backbone_display_name(backbone)

    subset = (
        data
        .sort_values(
            "importance_mean",
            ascending=False,
        )
        .head(top_k)
        .copy()
    )

    if subset.empty:
        return

    # --------------------------------------------------------
    # Préparation des informations
    # --------------------------------------------------------

    subset["rank"] = np.arange(
        1,
        len(subset) + 1,
    )

    total_importance = subset["importance_mean"].sum()

    subset["relative_pct"] = np.where(
        total_importance > 0,
        100.0
        * subset["importance_mean"]
        / total_importance,
        0.0,
    )

    subset["cumulative_pct"] = (
        subset["relative_pct"].cumsum()
    )

    # Inversion pour avoir la variable la plus importante en haut
    plot_data = subset.iloc[::-1].reset_index(drop=True)

    means = plot_data[
        "importance_mean"
    ].to_numpy(dtype=float)

    stds = plot_data[
        "importance_std"
    ].to_numpy(dtype=float)

    relative_pct = plot_data[
        "relative_pct"
    ].to_numpy(dtype=float)

    ranks = plot_data[
        "rank"
    ].to_numpy(dtype=int)

    features = plot_data[
        "feature"
    ].tolist()

    y_positions = np.arange(len(plot_data))

    labels = [
        f"{rank:02d}  {prettify_feature_name(feature)}"
        for rank, feature in zip(ranks, features)
    ]

    # --------------------------------------------------------
    # Taille dynamique
    # --------------------------------------------------------

    figure_height = max(
        8,
        0.52 * len(plot_data) + 2.5,
    )

    fig, ax = plt.subplots(
        figsize=(13.5, figure_height)
    )

    # Dégradé selon l'importance
    if means.max() > means.min():
        normalized_importance = (
            means - means.min()
        ) / (
            means.max() - means.min()
        )
    else:
        normalized_importance = np.ones_like(means)

    colors = plt.cm.viridis(
        0.20 + 0.75 * normalized_importance
    )

    # --------------------------------------------------------
    # Tiges du lollipop chart
    # --------------------------------------------------------

    for y_value, mean_value, color in zip(
        y_positions,
        means,
        colors,
    ):
        ax.hlines(
            y=y_value,
            xmin=0,
            xmax=mean_value,
            color=color,
            linewidth=3.2,
            alpha=0.78,
            zorder=2,
        )

    # --------------------------------------------------------
    # Points principaux
    # --------------------------------------------------------

    point_sizes = (
        100
        + 280
        * means
        / means.max()
        if means.max() > 0
        else np.full_like(means, 120)
    )

    ax.scatter(
        means,
        y_positions,
        s=point_sizes,
        c=colors,
        edgecolor="white",
        linewidth=1.4,
        zorder=4,
    )

    # Cercle supplémentaire sur la meilleure variable
    best_position = len(plot_data) - 1

    ax.scatter(
        means[best_position],
        y_positions[best_position],
        s=point_sizes[best_position] + 170,
        facecolor="none",
        edgecolor="black",
        linewidth=1.5,
        zorder=5,
    )

    # --------------------------------------------------------
    # Barres d'erreur
    # --------------------------------------------------------

    ax.errorbar(
        means,
        y_positions,
        xerr=stds,
        fmt="none",
        ecolor="black",
        elinewidth=1.0,
        capsize=3.5,
        capthick=1.0,
        alpha=0.55,
        zorder=3,
    )

    # --------------------------------------------------------
    # Annotations
    # --------------------------------------------------------

    maximum_extent = np.max(
        means + stds
    )

    annotation_margin = max(
        maximum_extent * 0.025,
        0.0005,
    )

    for y_value, mean_value, std_value, percentage in zip(
        y_positions,
        means,
        stds,
        relative_pct,
    ):
        ax.text(
            mean_value
            + std_value
            + annotation_margin,
            y_value,
            (
                f"{mean_value:.4f}"
                f"  ·  {percentage:.1f} %"
            ),
            va="center",
            ha="left",
            fontsize=8.7,
            fontweight="medium",
        )

    # --------------------------------------------------------
    # Étiquette de la meilleure variable
    # --------------------------------------------------------

    best_mean = means[best_position]
    best_std = stds[best_position]

    ax.annotate(
        "Variable la plus importante",
        xy=(
            best_mean,
            y_positions[best_position],
        ),
        xytext=(
            best_mean + maximum_extent * 0.28,
            y_positions[best_position] - 0.75,
        ),
        fontsize=9,
        fontweight="semibold",
        ha="left",
        va="center",
        bbox={
            "boxstyle": "round,pad=0.30",
            "facecolor": "white",
            "edgecolor": "black",
            "alpha": 0.90,
        },
        arrowprops={
            "arrowstyle": "->",
            "linewidth": 0.9,
            "color": "black",
        },
        zorder=6,
    )

    # --------------------------------------------------------
    # Axes
    # --------------------------------------------------------

    ax.set_yticks(y_positions)

    ax.set_yticklabels(
        labels,
        fontsize=9.2,
        linespacing=1.12,
    )

    ax.set_xlabel(
        "Importance moyenne XGBoost",
        fontsize=11.5,
        fontweight="semibold",
        labelpad=12,
    )

    ax.set_ylabel("")

    ax.set_title(
        f"Top-{len(plot_data)} des variables les plus discriminantes",
        fontsize=16,
        fontweight="bold",
        pad=24,
    )

    ax.text(
        0.5,
        1.01,
        (
            f"XGBoost associé à {display_name} · "
            f"protocole : {protocol} · "
            f"moyenne ± écart-type entre les seeds"
        ),
        transform=ax.transAxes,
        ha="center",
        va="bottom",
        fontsize=9.6,
        alpha=0.72,
    )

    # --------------------------------------------------------
    # Zone de mise en évidence du Top-5
    # --------------------------------------------------------

    top_5_start = max(
        len(plot_data) - 5 - 0.5,
        -0.5,
    )

    ax.axhspan(
        top_5_start,
        len(plot_data) - 0.5,
        alpha=0.055,
        zorder=0,
    )

    ax.text(
        0.012,
        0.985,
        "Top-5",
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=8.5,
        fontweight="bold",
        alpha=0.65,
    )

    # --------------------------------------------------------
    # Limites et format
    # --------------------------------------------------------

    right_limit = max(
        maximum_extent * 1.42,
        means.max() * 1.35,
    )

    ax.set_xlim(
        0,
        right_limit if right_limit > 0 else 1,
    )

    ax.xaxis.set_major_formatter(
        FuncFormatter(
            lambda value, _: (
                f"{value:.3f}"
                if abs(value) < 1
                else f"{value:.1f}"
            )
        )
    )

    # --------------------------------------------------------
    # Style scientifique
    # --------------------------------------------------------

    ax.grid(
        axis="x",
        linestyle="--",
        linewidth=0.7,
        alpha=0.24,
        zorder=0,
    )

    ax.set_axisbelow(True)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)

    ax.spines["bottom"].set_alpha(0.35)

    ax.tick_params(
        axis="y",
        length=0,
        pad=8,
    )

    ax.tick_params(
        axis="x",
        labelsize=9,
    )

    # --------------------------------------------------------
    # Notes de lecture
    # --------------------------------------------------------

    number_of_runs = int(
        plot_data["n_runs"].max()
    )

    ax.text(
        0.99,
        0.015,
        (
            f"Point : importance moyenne · "
            f"barre : ± écart-type · "
            f"n maximal = {number_of_runs} exécution(s)"
        ),
        transform=ax.transAxes,
        ha="right",
        va="bottom",
        fontsize=8,
        fontstyle="italic",
        alpha=0.62,
    )

    plt.tight_layout()

    output_name = (
        f"xgb_importance_lollipop_"
        f"{sanitize_filename(protocol)}_"
        f"{sanitize_filename(backbone)}"
    )

    save_figure(output_name)

    plt.show()
    plt.close(fig)


# ============================================================
# Génération des figures
# ============================================================

for (
    protocol,
    backbone,
), subset_group in importance_summary.groupby(
    ["protocol", "backbone"],
    sort=True,
    observed=True,
):
    plot_feature_importance_lollipop(
        data=subset_group,
        protocol=protocol,
        backbone=backbone,
        top_k=20,
    )

## 15 — Résumé final et fichiers utiles

In [ ]:
best_overall = summaries_df.loc[summaries_df["final_weighted_f1"].idxmax()]

comparison_summary = (
    summaries_df
    .groupby(["protocol", "backbone_display"])[[
        "final_weighted_f1",
        "final_macro_f1_present",
        "final_macro_f1_all",
        "final_mcc",
        "llm_train_seconds",
        "peak_vram_gib",
        "adapter_size_mb",
    ]]
    .agg(["mean", "std"])
)

print("Meilleur run global :")
display(best_overall.to_frame(name="value"))

print("\nRésumé comparatif :")
display(comparison_summary)

summary_export = {
    "best_protocol": best_overall["protocol"],
    "best_backbone": best_overall["backbone_display"],
    "best_seed": int(best_overall["seed"]),
    "selected_candidate": best_overall["selected_candidate"],
    "weighted_f1": float(best_overall["final_weighted_f1"]),
    "macro_f1_present": float(best_overall["final_macro_f1_present"]),
    "macro_f1_all": float(best_overall["final_macro_f1_all"]),
    "mcc": float(best_overall["final_mcc"]),
}

with open(
    TABLE_DIR / "best_comparative_summary.json",
    "w",
    encoding="utf-8",
) as handle:
    json.dump(summary_export, handle, indent=2, ensure_ascii=False)

print("Figures :", FIG_DIR)
print("Tableaux :", TABLE_DIR)
print("Modèles :", MODEL_DIR)

## 16 — Analyse conforme et explicabilité par SLM

In [ ]:

trust_summary_rows = []
trust_class_coverage_rows = []
xai_detail_rows = []
verbalizer_rows = []

for backbone in CFG.backbones:
    for protocol in CFG.protocols:
        protocol_runs = summaries_df[
            (summaries_df["protocol"] == protocol)
            & (summaries_df["backbone"] == backbone)
        ].copy()
        if protocol_runs.empty:
            continue
        best_idx = protocol_runs["final_weighted_f1"].idxmax()
        best_row = protocol_runs.loc[best_idx]
        seed = int(best_row["seed"])
        run_dir = MODEL_DIR / f"{protocol}__{backbone}__seed_{seed}"

        print("\n" + "=" * 90)
        print(
            f"TRUSTWORTHINESS : {backbone_display_name(backbone)} | "
            f"{protocol} | seed={seed}"
        )
        print("=" * 90)

        splits = make_splits(
            data=df,
            protocol=protocol,
            seed=seed,
            label_col=LABEL_COL,
            train_fraction=CFG.train_fraction,
            val_fraction=CFG.val_fraction,
            cal_fraction=CFG.cal_fraction,
            test_fraction=CFG.test_fraction,
        )
        context = transform_splits_from_saved_preprocessing(splits, run_dir)
        arrays = context["arrays"]
        class_names = context["class_names"]
        n_classes = context["n_classes"]
        numeric_features = context["numeric_features"]
        text_features = context["text_features"]

        with open(run_dir / "run_metadata.json", "r", encoding="utf-8") as handle:
            metadata_dict = json.load(handle)
        selected_candidate = metadata_dict["selected_candidate"]
        best_alpha = float(metadata_dict["best_alpha_numeric"])

        xgb_model = joblib.load(run_dir / "xgboost_model.joblib")
        stacker = joblib.load(run_dir / "fusion_stacker.joblib")
        slm_model, slm_tokenizer = load_slm_adapter(
            run_dir,
            n_classes,
            class_names,
            metadata_dict["model_checkpoint"],
        )

        probabilities = {}
        for split_name in ["cal", "test"]:
            loader = make_full_llm_loader(
                arrays,
                split_name,
                slm_tokenizer,
                text_features,
                CFG,
            )
            llm_probs, labels, _ = collect_llm_probs(slm_model, loader, DEVICE)
            xgb_probs = np.asarray(xgb_model.predict_proba(arrays[split_name]["raw_numeric"]), dtype=np.float64)
            xgb_probs /= xgb_probs.sum(axis=1, keepdims=True)
            final_probs = get_candidate_probabilities(
                selected_candidate,
                llm_probs,
                xgb_probs,
                stacker,
                best_alpha,
                n_classes,
            )
            probabilities[split_name] = {
                "llm": llm_probs,
                "xgb": xgb_probs,
                "final": final_probs,
                "labels": labels,
            }

        partition = np.load(run_dir / "calibration_partition_indices.npz")
        selection_idx = partition["selection_indices"]
        conformal_idx = partition["conformal_indices"]

        temperature = fit_temperature(
            probabilities["cal"]["final"][selection_idx],
            probabilities["cal"]["labels"][selection_idx],
        )
        cal_probs_temp = apply_temperature(probabilities["cal"]["final"][conformal_idx], temperature)
        test_probs_temp = apply_temperature(probabilities["test"]["final"], temperature)
        cal_labels_conf = probabilities["cal"]["labels"][conformal_idx]
        test_labels = probabilities["test"]["labels"]

        method_sets = {}
        for method in ["APS", "RAPS"]:
            scores = conformal_true_scores(
                cal_probs_temp,
                cal_labels_conf,
                method=method,
                raps_lambda=CFG.raps_lambda,
                k_reg=CFG.raps_k_reg,
            )
            qhat = conformal_quantile(scores, CFG.conformal_alpha)
            prediction_sets = build_prediction_sets(
                test_probs_temp,
                qhat=qhat,
                method=method,
                raps_lambda=CFG.raps_lambda,
                k_reg=CFG.raps_k_reg,
            )
            overall, classwise = evaluate_prediction_sets(
                prediction_sets,
                test_labels,
                class_names,
                CFG.conformal_abstain_size,
            )
            method_sets[method] = prediction_sets
            trust_summary_rows.append({
                "protocol": protocol,
                "backbone": backbone,
                "backbone_display": backbone_display_name(backbone),
                "seed": seed,
                "selected_candidate": selected_candidate,
                "temperature": temperature,
                "method": method,
                "qhat": qhat,
                **overall,
            })
            classwise["protocol"] = protocol
            classwise["seed"] = seed
            classwise["method"] = method
            trust_class_coverage_rows.append(classwise)

        # RAPS est utilisé pour les explications finales.
        raps_sets = method_sets["RAPS"]
        explanation_indices = stratified_explanation_indices(
            test_labels,
            CFG.explanation_samples,
            seed + 71,
        )

        X_explain = arrays["test"]["raw_numeric"][explanation_indices]
        shap_values = get_tree_shap_values(xgb_model, X_explain, n_classes)
        final_predictions = test_probs_temp.argmax(axis=1)
        train_medians = np.median(arrays["train"]["raw_numeric"], axis=0)

        slm_prompts = []
        slm_row_indices = []
        templates = []

        for local_position, test_index in enumerate(tqdm(explanation_indices, desc=f"XAI {protocol}")):
            predicted_class = int(final_predictions[test_index])
            predicted_label = class_names[predicted_class]
            confidence = float(test_probs_temp[test_index, predicted_class])

            # TreeSHAP
            class_shap = shap_values[local_position, predicted_class]
            shap_order = np.argsort(-np.abs(class_shap))[:CFG.explanation_top_k]
            shap_evidence = [(numeric_features[j], float(class_shap[j])) for j in shap_order]

            modified_numeric = arrays["test"]["raw_numeric"][test_index].copy()
            modified_numeric[shap_order] = train_medians[shap_order]
            numeric_after = xgb_model.predict_proba(modified_numeric[None, :])[0]
            xgb_original_class_prob = float(probabilities["test"]["xgb"][test_index, predicted_class])
            shap_fidelity_drop = xgb_original_class_prob - float(numeric_after[predicted_class])

            # Occlusion MiniLM
            original_text_values = arrays["test"]["scaled_text"][test_index].copy()
            variants = np.repeat(original_text_values[None, :], len(text_features), axis=0)
            variants[np.arange(len(text_features)), np.arange(len(text_features))] = 0.0
            start_occ = time.perf_counter()
            variant_probs = slm_predict_values(slm_model, slm_tokenizer, text_features, variants)
            occlusion_seconds = time.perf_counter() - start_occ
            llm_original_prob = float(probabilities["test"]["llm"][test_index, predicted_class])
            drops = llm_original_prob - variant_probs[:, predicted_class]
            occ_order = np.argsort(-drops)[:CFG.explanation_top_k]
            occlusion_evidence = [(text_features[j], float(drops[j])) for j in occ_order]
            combined_values = original_text_values.copy()
            combined_values[occ_order] = 0.0
            combined_prob = slm_predict_values(
                slm_model,
                slm_tokenizer,
                text_features,
                combined_values[None, :],
            )[0, predicted_class]
            occlusion_fidelity_drop = llm_original_prob - float(combined_prob)

            shap_names = [name for name, _ in shap_evidence]
            occ_names = [name for name, _ in occlusion_evidence]
            agreement = len(set(shap_names) & set(occ_names)) / max(1, len(set(shap_names) | set(occ_names)))

            conformal_labels = [class_names[index] for index in raps_sets[test_index]]
            template = deterministic_template_explanation(
                predicted_label,
                confidence,
                conformal_labels,
                shap_evidence,
                occlusion_evidence,
            )
            templates.append(template)

            allowed_features = list(dict.fromkeys(shap_names + occ_names))
            prompt = (
                f"Write exactly three short {CFG.explanation_language} sentences for a cybersecurity analyst. "
                "Use only the supplied evidence. Cite feature names exactly in square brackets. "
                "Do not invent causes, protocols, ports, or features. "
                f"Prediction: {predicted_label}. Confidence: {confidence:.3f}. "
                f"Conformal alternatives: {', '.join(conformal_labels)}. "
                f"Numeric evidence: {', '.join(f'[{n}] contribution {v:+.3f}' for n, v in shap_evidence)}. "
                f"MiniLM occlusion evidence: {', '.join(f'[{n}] drop {v:.3f}' for n, v in occlusion_evidence)}. "
                "If the conformal set has more than one class, explicitly state that the decision is uncertain."
            )
            slm_prompts.append(prompt)
            slm_row_indices.append(len(xai_detail_rows))

            xai_detail_rows.append({
                "protocol": protocol,
                "backbone": backbone,
                "backbone_display": backbone_display_name(backbone),
                "seed": seed,
                "test_index": int(test_index),
                "true_label": class_names[int(test_labels[test_index])],
                "predicted_label": predicted_label,
                "confidence": confidence,
                "conformal_set": " | ".join(conformal_labels),
                "tree_shap_features": " | ".join(shap_names),
                "tree_shap_values": " | ".join(f"{value:.6f}" for _, value in shap_evidence),
                "tree_shap_fidelity_drop": shap_fidelity_drop,
                "occlusion_features": " | ".join(occ_names),
                "occlusion_drops": " | ".join(f"{value:.6f}" for _, value in occlusion_evidence),
                "occlusion_fidelity_drop": occlusion_fidelity_drop,
                "occlusion_latency_seconds": occlusion_seconds,
                "cross_method_jaccard": agreement,
                "template_explanation": template,
                "allowed_features": " | ".join(allowed_features),
            })

        slm_count = min(CFG.slm_explanation_samples, len(slm_prompts))
        generated, generation_seconds = generate_slm_explanations(slm_prompts[:slm_count])
        per_slm_seconds = generation_seconds / max(1, slm_count)

        protocol_xai_start = len(xai_detail_rows) - len(explanation_indices)
        for offset, row_index in enumerate(range(protocol_xai_start, protocol_xai_start + len(explanation_indices))):
            row = xai_detail_rows[row_index]
            allowed = row["allowed_features"].split(" | ") if row["allowed_features"] else []
            template_metrics = citation_metrics(row["template_explanation"], allowed)
            verbalizer_rows.append({
                "protocol": protocol,
                "backbone": backbone,
                "backbone_display": backbone_display_name(backbone),
                "seed": seed,
                "test_index": row["test_index"],
                "method": "Deterministic template",
                "text": row["template_explanation"],
                "latency_seconds": 0.0,
                "word_count": len(row["template_explanation"].split()),
                "predicted_label_present": float(row["predicted_label"].lower() in row["template_explanation"].lower()),
                **template_metrics,
            })
            if offset < slm_count:
                slm_text = generated[offset]
                slm_metrics = citation_metrics(slm_text, allowed)
                verbalizer_rows.append({
                    "protocol": protocol,
                "backbone": backbone,
                "backbone_display": backbone_display_name(backbone),
                    "seed": seed,
                    "test_index": row["test_index"],
                    "method": "FLAN-T5-small",
                    "text": slm_text,
                    "latency_seconds": per_slm_seconds,
                    "word_count": len(slm_text.split()),
                    "predicted_label_present": float(row["predicted_label"].lower() in slm_text.lower()),
                    **slm_metrics,
                })

        del slm_model
        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    trust_summary_df = pd.DataFrame(trust_summary_rows)
    trust_class_coverage_df = pd.concat(trust_class_coverage_rows, ignore_index=True)
    xai_details_df = pd.DataFrame(xai_detail_rows)
    verbalizer_comparison_df = pd.DataFrame(verbalizer_rows)

    trust_summary_df.to_csv(TABLE_DIR / "conformal_summary.csv", index=False)
    trust_class_coverage_df.to_csv(TABLE_DIR / "conformal_classwise_coverage.csv", index=False)
    xai_details_df.to_csv(TABLE_DIR / "explainability_details.csv", index=False)
    verbalizer_comparison_df.to_csv(TABLE_DIR / "slm_verbalizer_comparison.csv", index=False)


xai_method_rows = []
for (protocol, backbone), subset in xai_details_df.groupby(
    ["protocol", "backbone"]
):
    xai_method_rows.extend([
        {
            "protocol": protocol,
            "backbone": backbone,
            "method": "TreeSHAP",
            "mean_fidelity_drop": subset["tree_shap_fidelity_drop"].mean(),
            "mean_latency_seconds": np.nan,
            "mean_cross_method_jaccard": subset["cross_method_jaccard"].mean(),
        },
        {
            "protocol": protocol,
            "backbone": backbone,
            "method": f"{backbone_display_name(backbone)} feature occlusion",
            "mean_fidelity_drop": subset["occlusion_fidelity_drop"].mean(),
            "mean_latency_seconds": subset["occlusion_latency_seconds"].mean(),
            "mean_cross_method_jaccard": subset["cross_method_jaccard"].mean(),
        },
    ])

xai_method_summary_df = pd.DataFrame(xai_method_rows)
xai_method_summary_df.to_csv(
    TABLE_DIR / "xai_method_comparison.csv",
    index=False,
)

display(trust_summary_df)
display(xai_method_summary_df)
display(
    verbalizer_comparison_df
    .groupby(["backbone_display", "method"])
    .agg({
        "valid_citation_rate": "mean",
        "unsupported_citation_rate": "mean",
        "evidence_coverage": "mean",
        "latency_seconds": "mean",
        "word_count": "mean",
        "predicted_label_present": "mean",
    })
)


## 17 — Graphiques de conformité et d’explicabilité

In [ ]:
# Couverture vs taille moyenne
for (protocol, backbone), subset in trust_summary_df.groupby(
    ["protocol", "backbone"]
):
    plt.figure(figsize=(8, 5))
    for _, row in subset.iterrows():
        plt.scatter(row["average_set_size"], row["coverage"], s=120)
        plt.annotate(
            row["method"],
            (row["average_set_size"], row["coverage"]),
            xytext=(5, 5),
            textcoords="offset points",
        )
    plt.axhline(
        1.0 - CFG.conformal_alpha,
        linestyle="--",
        label="Couverture cible",
    )
    plt.xlabel("Taille moyenne de l'ensemble")
    plt.ylabel("Couverture")
    plt.ylim(0.0, 1.02)
    plt.title(
        f"Conformité — {backbone_display_name(backbone)} | {protocol}"
    )
    plt.legend()
    save_figure(f"conformal_{protocol}_{backbone}")
    plt.show()

# Couverture RAPS par classe
for (protocol, backbone), subset_all in trust_class_coverage_df.groupby(
    ["protocol", "backbone"]
):
    subset = subset_all[
        (subset_all["method"] == "RAPS")
        & (subset_all["support"] > 0)
    ]
    plt.figure(figsize=(12, 5))
    plt.bar(subset["class"], subset["coverage"])
    plt.axhline(1.0 - CFG.conformal_alpha, linestyle="--")
    plt.xticks(rotation=45, ha="right")
    plt.ylim(0.0, 1.02)
    plt.ylabel("Couverture RAPS")
    plt.title(
        f"Couverture conforme par classe — "
        f"{backbone_display_name(backbone)} | {protocol}"
    )
    save_figure(f"raps_classwise_{protocol}_{backbone}")
    plt.show()

# Fidélité comparative XAI
xai_plot = (
    xai_method_summary_df
    .groupby(["backbone", "method"], as_index=False)["mean_fidelity_drop"]
    .mean()
)
plt.figure(figsize=(12, 6))
labels = [
    f"{backbone_display_name(row['backbone'])}\n{row['method']}"
    for _, row in xai_plot.iterrows()
]
plt.bar(labels, xai_plot["mean_fidelity_drop"])
plt.ylabel("FidelityDrop@k")
plt.title("Fidélité comparative des explications")
plt.xticks(rotation=25, ha="right")
save_figure("comparative_xai_faithfulness")
plt.show()

# Verbaliseurs
verbalizer_agg = (
    verbalizer_comparison_df
    .groupby(["backbone", "method"], as_index=False)
    .agg({
        "valid_citation_rate": "mean",
        "unsupported_citation_rate": "mean",
        "evidence_coverage": "mean",
        "predicted_label_present": "mean",
        "latency_seconds": "mean",
    })
)
verbalizer_agg["groundedness"] = (
    1.0 - verbalizer_agg["unsupported_citation_rate"]
)
display(verbalizer_agg)

## 18 — Conseils d’utilisation pour l’article

### Mesures à rapporter
- **Weighted-F1**
- **Macro-F1**
- **MCC**
- **Balanced Accuracy**
- temps d’entraînement
- VRAM
- taux de troncature

### Figures déjà prêtes
- `global_class_distribution`
- `performance_weighted_f1_by_protocol`
- `performance_macro_f1_by_protocol`
- `performance_mcc_by_protocol`
- `efficiency_vs_weighted_f1`
- `training_loss_<protocol>`
- `training_macro_f1_bal_<protocol>`
- `token_length_audit_<protocol>`
- `split_distribution_<protocol>`
- `confusion_matrix_best_<protocol>`
- `xgb_feature_importance_<protocol>`

### Lecture correcte du résultat
Si la fusion sélectionnée est `xgboost`, `fusion_alpha_1.000` ou `stacking` avec `alpha=1.0`, cela signifie que la branche numérique porte l’essentiel de la performance.
Dans ce cas, MiniLM doit être présenté comme **branche sémantique frugale complémentaire**, et non comme source unique du score final.

### Modules de confiance v6
- APS et RAPS sont comparés sur une moitié de `CAL` réservée exclusivement à la conformité.
- TreeSHAP explique la branche XGBoost.
- L’occlusion explique la branche MiniLM.
- FLAN-T5-small ne décide pas de la classe : il verbalise uniquement les preuves déjà calculées.
- Le taux de citations non supportées doit être rapporté comme mesure d’hallucination explicative.


## 19 — Comparaison finale MiniLM–ModernBERT

Cette section produit les tableaux et figures utilisables directement dans
l'article :

- performance moyenne ± écart-type ;
- Macro-F1 sur classes présentes et sur les dix classes ;
- coûts de calcul ;
- efficacité LoRA ;
- visualisations circulaires ;
- radars multi-objectifs ;
- tests appariés entre backbones.

In [ ]:
article_table = (
    summaries_df
    .groupby(["protocol", "backbone_display"])[[
        "final_accuracy",
        "final_weighted_f1",
        "final_macro_f1_present",
        "final_macro_f1_all",
        "final_balanced_accuracy",
        "final_mcc",
        "final_label_space_coverage",
        "llm_train_seconds",
        "peak_vram_gib",
        "adapter_size_mb",
    ]]
    .agg(["mean", "std"])
)
article_table.to_csv(
    TABLE_DIR / "article_minilm_modernbert_comparison.csv"
)
display(article_table)

In [ ]:
# Comparaisons appariées MiniLM vs ModernBERT.
paired_rows = []

for protocol in CFG.protocols:
    subset = summaries_df[summaries_df["protocol"] == protocol]
    for metric in [
        "final_weighted_f1",
        "final_macro_f1_present",
        "final_macro_f1_all",
        "final_mcc",
        "llm_train_seconds",
        "peak_vram_gib",
    ]:
        pivot = subset.pivot(
            index="seed",
            columns="backbone",
            values=metric,
        )
        if not {"minilm", "modernbert"}.issubset(pivot.columns):
            continue
        paired = pivot[["minilm", "modernbert"]].dropna()

        if wilcoxon is not None and len(paired) >= 2:
            try:
                statistic, pvalue = wilcoxon(
                    paired["modernbert"],
                    paired["minilm"],
                    alternative="two-sided",
                )
            except Exception:
                statistic, pvalue = np.nan, np.nan
        else:
            statistic, pvalue = np.nan, np.nan

        paired_rows.append({
            "protocol": protocol,
            "metric": metric,
            "n_pairs": len(paired),
            "minilm_mean": paired["minilm"].mean(),
            "modernbert_mean": paired["modernbert"].mean(),
            "modernbert_minus_minilm": (
                paired["modernbert"] - paired["minilm"]
            ).mean(),
            "wilcoxon_statistic": statistic,
            "wilcoxon_pvalue": pvalue,
        })

paired_comparison_df = pd.DataFrame(paired_rows)
paired_comparison_df.to_csv(
    TABLE_DIR / "paired_minilm_modernbert_tests.csv",
    index=False,
)
display(paired_comparison_df)

In [ ]:
# Graphique principal : performance par backbone.
metric_columns = [
    ("final_weighted_f1", "Weighted-F1"),
    ("final_macro_f1_present", "Macro-F1 present"),
    ("final_macro_f1_all", "Macro-F1 all"),
    ("final_mcc", "MCC"),
]

for metric, label in metric_columns:
    grouped = (
        summaries_df
        .groupby(["protocol", "backbone_display"])[metric]
        .agg(["mean", "std"])
        .reset_index()
    )

    scenarios = grouped["protocol"].unique().tolist()
    x = np.arange(len(scenarios))
    width = 0.35

    plt.figure(figsize=(10, 5.5))
    for i, backbone_name in enumerate(
        ["MiniLM", "ModernBERT"]
    ):
        subset = (
            grouped[grouped["backbone_display"] == backbone_name]
            .set_index("protocol")
            .reindex(scenarios)
        )
        plt.bar(
            x + (i - 0.5) * width,
            subset["mean"],
            width=width,
            yerr=subset["std"].fillna(0.0),
            capsize=4,
            label=backbone_name,
        )

    plt.xticks(x, scenarios)
    plt.ylim(0.0, 1.02)
    plt.ylabel(label)
    plt.title(f"{label} — MiniLM vs ModernBERT")
    plt.legend()
    save_figure(f"backbone_comparison_{metric}")
    plt.show()

In [ ]:
# Donuts : proportion de paramètres entraînables.
best_by_backbone = (
    summaries_df
    .sort_values("final_weighted_f1", ascending=False)
    .groupby("backbone")
    .head(1)
)

fig, axes = plt.subplots(
    1,
    len(best_by_backbone),
    figsize=(6 * len(best_by_backbone), 5),
)
if len(best_by_backbone) == 1:
    axes = [axes]

for ax, (_, row) in zip(axes, best_by_backbone.iterrows()):
    trainable = float(row["llm_trainable_parameters"])
    frozen = float(
        row["llm_total_parameters"]
        - row["llm_trainable_parameters"]
    )
    ax.pie(
        [trainable, frozen],
        labels=["LoRA + tête", "Paramètres gelés"],
        startangle=90,
        wedgeprops={"width": 0.42, "edgecolor": "white"},
        autopct="%1.2f%%",
    )
    ax.set_title(
        f"{row['backbone_display']}\n"
        f"{row['llm_trainable_percent']:.3f}% entraînable"
    )

plt.tight_layout()
save_figure("donut_trainable_parameters_by_backbone")
plt.show()

In [ ]:
# Donut charts: proportion of trainable parameters.
best_by_backbone = (
    summaries_df
    .sort_values("final_weighted_f1", ascending=False)
    .groupby("backbone")
    .head(1)
)

fig, axes = plt.subplots(
    1,
    len(best_by_backbone),
    figsize=(6 * len(best_by_backbone), 5),
)

if len(best_by_backbone) == 1:
    axes = [axes]

for ax, (_, row) in zip(axes, best_by_backbone.iterrows()):
    trainable = float(row["llm_trainable_parameters"])
    frozen = float(
        row["llm_total_parameters"]
        - row["llm_trainable_parameters"]
    )

    ax.pie(
        [trainable, frozen],
        labels=["LoRA + head", "Frozen parameters"],
        startangle=90,
        wedgeprops={"width": 0.42, "edgecolor": "white"},
        autopct="%1.2f%%",
    )

    ax.set_title(
        f"{row['backbone_display']}\n"
        f"{row['llm_trainable_percent']:.3f}% trainable"
    )

plt.tight_layout()
save_figure("donut_trainable_parameters_by_backbone")
plt.show()

In [ ]:
# Donuts : couverture de l'espace des labels par protocole.
best_scenarios = (
    summaries_df
    .sort_values("final_weighted_f1", ascending=False)
    .groupby(["protocol", "backbone"])
    .head(1)
)

fig, axes = plt.subplots(
    len(CFG.protocols),
    len(CFG.backbones),
    figsize=(11, 5 * len(CFG.protocols)),
)
axes = np.asarray(axes)
if axes.ndim == 1:
    axes = axes.reshape(len(CFG.protocols), len(CFG.backbones))

for i, protocol in enumerate(CFG.protocols):
    for j, backbone in enumerate(CFG.backbones):
        ax = axes[i, j]
        subset = best_scenarios[
            (best_scenarios["protocol"] == protocol)
            & (best_scenarios["backbone"] == backbone)
        ]
        if subset.empty:
            ax.axis("off")
            continue
        row = subset.iloc[0]
        present = int(row["final_present_class_count"])
        absent = int(row["n_classes"] - present)
        ax.pie(
            [present, absent],
            labels=["Présentes", "Absentes"],
            startangle=90,
            wedgeprops={"width": 0.42, "edgecolor": "white"},
            autopct="%1.0f%%",
        )
        ax.set_title(
            f"{backbone_display_name(backbone)} | {protocol}"
        )

plt.tight_layout()
save_figure("donut_label_coverage_all_scenarios")
plt.show()

In [ ]:
# Donut charts: label-space coverage by protocol.
best_scenarios = (
    summaries_df
    .sort_values("final_weighted_f1", ascending=False)
    .groupby(["protocol", "backbone"])
    .head(1)
)

fig, axes = plt.subplots(
    len(CFG.protocols),
    len(CFG.backbones),
    figsize=(11, 5 * len(CFG.protocols)),
)

axes = np.asarray(axes)

if axes.ndim == 1:
    axes = axes.reshape(
        len(CFG.protocols),
        len(CFG.backbones),
    )

for i, protocol in enumerate(CFG.protocols):
    for j, backbone in enumerate(CFG.backbones):
        ax = axes[i, j]

        subset = best_scenarios[
            (best_scenarios["protocol"] == protocol)
            & (best_scenarios["backbone"] == backbone)
        ]

        if subset.empty:
            ax.axis("off")
            continue

        row = subset.iloc[0]

        present = int(row["final_present_class_count"])
        absent = int(row["n_classes"] - present)

        ax.pie(
            [present, absent],
            labels=["Present", "Absent"],
            startangle=90,
            wedgeprops={
                "width": 0.42,
                "edgecolor": "white",
            },
            autopct="%1.0f%%",
        )

        ax.set_title(
            f"{backbone_display_name(backbone)} | {protocol}"
        )

plt.tight_layout()
save_figure("donut_label_coverage_all_scenarios")
plt.show()

In [ ]:
# Radar performance/efficacité par backbone.
def radar_plot(ax, values_by_name, categories, title):
    angles = np.linspace(
        0,
        2 * np.pi,
        len(categories),
        endpoint=False,
    ).tolist()
    angles += angles[:1]

    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories)
    ax.set_ylim(0.0, 1.0)

    for name, values in values_by_name.items():
        values_closed = list(values) + [values[0]]
        ax.plot(angles, values_closed, linewidth=2, label=name)
        ax.fill(angles, values_closed, alpha=0.08)

    ax.set_title(title, y=1.12)

radar_agg = (
    summaries_df
    .groupby(["protocol", "backbone_display"])
    .agg({
        "final_weighted_f1": "mean",
        "final_macro_f1_present": "mean",
        "final_macro_f1_all": "mean",
        "final_mcc": "mean",
        "llm_train_seconds": "mean",
        "peak_vram_gib": "mean",
    })
    .reset_index()
)

max_time = radar_agg["llm_train_seconds"].max()
max_vram = radar_agg["peak_vram_gib"].max()
radar_agg["speed_score"] = (
    1.0 - radar_agg["llm_train_seconds"] / max_time
)
radar_agg["memory_score"] = (
    1.0 - radar_agg["peak_vram_gib"] / max_vram
)

categories = [
    "Weighted-F1",
    "Macro-F1 present",
    "Macro-F1 all",
    "MCC",
    "Speed",
    "Memory",
]

fig, axes = plt.subplots(
    1,
    len(CFG.protocols),
    figsize=(7 * len(CFG.protocols), 6),
    subplot_kw={"projection": "polar"},
)
if len(CFG.protocols) == 1:
    axes = [axes]

for ax, protocol in zip(axes, CFG.protocols):
    subset = radar_agg[radar_agg["protocol"] == protocol]
    values_by_name = {}
    for _, row in subset.iterrows():
        values_by_name[row["backbone_display"]] = [
            row["final_weighted_f1"],
            row["final_macro_f1_present"],
            row["final_macro_f1_all"],
            row["final_mcc"],
            row["speed_score"],
            row["memory_score"],
        ]
    radar_plot(
        ax,
        values_by_name,
        categories,
        f"Perf.–Eff | {protocol}",
    )

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=2)
#plt.tight_layout(rect=[0, 0, 0.2, 1.99])
save_figure("radar_minilm_modernbert_performance_efficiency")
plt.show()

In [ ]:
# Graphique bulles : coût, mémoire, paramètres et performance.
efficiency_best = (
    summaries_df
    .sort_values("final_weighted_f1", ascending=False)
    .groupby(["protocol", "backbone"])
    .head(1)
)

plt.figure(figsize=(11, 7))
for backbone in CFG.backbones:
    subset = efficiency_best[
        efficiency_best["backbone"] == backbone
    ]
    plt.scatter(
        subset["llm_train_seconds"] / 60.0,
        subset["final_macro_f1_all"],
        s=(
            subset["peak_vram_gib"]
            * 300
            + subset["llm_trainable_percent"] * 100
        ),
        label=backbone_display_name(backbone),
        alpha=0.75,
    )
    for _, row in subset.iterrows():
        plt.annotate(
            row["protocol"],
            (
                row["llm_train_seconds"] / 60.0,
                row["final_macro_f1_all"],
            ),
            xytext=(5, 5),
            textcoords="offset points",
        )

plt.xlabel("Temps d'entraînement (minutes)")
plt.ylabel("Macro-F1 all")
plt.ylim(0.0, 1.02)
plt.title(
    "Efficacité comparative des SLM "
    "(taille des bulles = mémoire + paramètres entraînables)"
)
plt.legend()
save_figure("bubble_slm_efficiency_comparison")
plt.show()

In [ ]:
# Comparaison de la conformité et de l'explicabilité par backbone.
conformal_backbone_table = (
    trust_summary_df
    .groupby(["protocol", "backbone_display", "method"])[[
        "coverage",
        "average_set_size",
        "singleton_rate",
        "abstention_rate",
    ]]
    .mean()
)
conformal_backbone_table.to_csv(
    TABLE_DIR / "conformal_backbone_comparison.csv"
)

xai_backbone_table = (
    xai_method_summary_df
    .groupby(["protocol", "backbone", "method"])[[
        "mean_fidelity_drop",
        "mean_latency_seconds",
        "mean_cross_method_jaccard",
    ]]
    .mean()
)
xai_backbone_table.to_csv(
    TABLE_DIR / "xai_backbone_comparison.csv"
)

display(conformal_backbone_table)
display(xai_backbone_table)

## 20 — Lecture scientifique recommandée

La comparaison doit répondre à quatre questions :

1. **ModernBERT améliore-t-il le Macro-F1 ou la stabilité ?**
2. **Le gain éventuel justifie-t-il son coût supplémentaire ?**
3. **Quel backbone produit les explications les plus fidèles et stables ?**
4. **Quel backbone fournit les ensembles conformes les plus petits à couverture comparable ?**

Le meilleur backbone ne doit pas être choisi uniquement selon le
Weighted-F1. Le choix final doit considérer :

- Macro-F1 sur les classes présentes ;
- Macro-F1 sur les dix classes ;
- MCC ;
- stabilité entre graines ;
- temps ;
- mémoire ;
- taille de l'adaptateur ;
- fidélité XAI ;
- efficacité APS/RAPS.